# **Metriche SOLR - DSpace**

## **Setup**

### Install requirements

In [ ]:
# Requirements are installed from ../requirements.txt outside notebook execution.


### Import packages and configuration loading

In [ ]:
import os
from datetime import date, timedelta
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
import yaml
import urllib3
from sshtunnel import SSHTunnelForwarder

EXPORT_SOURCE = "solr"
MONTHLY_EXPORT_FILES = {
    "solr_accesses_from_search_engines_reference_month.csv",
    "solr_bot_events_reference_month_by_type.csv",
    "solr_downloads_by_community_reference_month.csv",
    "solr_downloads_by_license_reference_month.csv",
    "solr_downloads_by_mimetype_reference_month.csv",
    "solr_downloads_by_resource_type_reference_month.csv",
    "solr_internal_events_192_168_reference_month_by_type.csv",
    "solr_internal_external_events_ratio_reference_month_by_bot_status_and_type.csv",
    "solr_most_frequent_bot_user_agents_reference_month.csv",
    "solr_most_frequent_ips_dns_reference_month.csv",
    "solr_most_frequent_user_agents_reference_month_by_bot_status.csv",
    "solr_most_searched_terms_reference_month.csv",
    "solr_top_downloaded_items_reference_month.csv",
    "solr_top_items_reference_month.csv",
    "solr_top_referrers_reference_month.csv",
    "solr_total_oai_records_reference_month.csv",
    "solr_total_searches_details_reference_month.csv",
    "solr_total_searches_reference_month_by_bot_status.csv",
    "solr_views_by_collection_reference_month.csv",
    "solr_views_by_current_community_reference_month.csv",
    "solr_views_downloads_ratio_by_item_reference_month.csv",
}


def load_config(config_path=None):
    config_path = config_path or os.environ.get(
        "DSPACE_REPORTING_CONFIG",
        "../config/settings.local.yaml",
    )
    with open(config_path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def next_month_start(month_label):
    year, month = map(int, month_label.split("-"))
    if month == 12:
        return date(year + 1, 1, 1)
    return date(year, month + 1, 1)


def get_scoped_export_dir(scope):
    export_root = PROJECT_ROOT / config["paths"].get("exports_root", "exports")
    base_dir = export_root / EXPORT_SOURCE / scope
    if config.get("run", {}).get("create_month_subfolders", True):
        return base_dir / REPORT_MONTH
    return base_dir


def export_path(filename):
    scope = "monthly" if filename in MONTHLY_EXPORT_FILES else "always"
    output_path = get_scoped_export_dir(scope) / filename
    if EXPORT_OUTPUTS:
        output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path


config = load_config()

REPORT_MONTH = config["run"]["reference_month"]
REPORT_MONTH_DATE = f"{REPORT_MONTH}-01"
REPORT_MONTH_END_DATE = next_month_start(REPORT_MONTH).isoformat()
REPORT_AS_OF_DATE = (next_month_start(REPORT_MONTH) - timedelta(days=1)).isoformat()
OVERWRITE_EXPORTS = config.get("run", {}).get("overwrite_exports", False)
EXPORT_OUTPUTS = config.get("run", {}).get("export_outputs", True)
ENVIRONMENT = config["project"]["environment"]
TIMEZONE = config["project"]["timezone"]

PROJECT_ROOT = Path.cwd().parent
EXPORT_ROOT = PROJECT_ROOT / config["paths"].get("exports_root", "exports")
source_export_dir = EXPORT_ROOT / EXPORT_SOURCE
monthly_export_dir = get_scoped_export_dir("monthly")
always_export_dir = get_scoped_export_dir("always")
export_dir = source_export_dir

SOLR_BASE_URL = config["solr"]["base_url"].rstrip("/")
SOLR_VERIFY_SSL = config["solr"].get("verify_ssl", True)
SOLR_TIMEOUT = config["solr"].get("timeout", 30)

SOLR_CORES = {
    "statistics": config["solr"].get("statistics_core", "statistics"),
    "search": config["solr"].get("search_core", "search"),
    "oai": config["solr"].get("oai_core", "oai"),
}

if not SOLR_VERIFY_SSL:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

_original_to_csv = pd.DataFrame.to_csv


def guarded_to_csv(self, path_or_buf=None, *args, **kwargs):
    if path_or_buf is not None and not EXPORT_OUTPUTS:
        output_path = Path(path_or_buf)
        print(f"Display-only mode, export skipped: {output_path.resolve()}")
        return None
    if path_or_buf is not None and not OVERWRITE_EXPORTS:
        output_path = Path(path_or_buf)
        if output_path.exists():
            print(f"Export exists and overwrite_exports=false, skipped: {output_path.resolve()}")
            return None
    return _original_to_csv(self, path_or_buf, *args, **kwargs)


pd.DataFrame.to_csv = guarded_to_csv

### SSH Tunnel

In [ ]:
# =========================================================
# SOLR CONNECTION CHECK
# =========================================================

print("=== Context of implementation ===")
print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Environment: {ENVIRONMENT}")
print(f"Timezone: {TIMEZONE}")
print(f"Reference month: {REPORT_MONTH}")
print(f"Export directory: {export_dir.resolve()}")
print()

SOLR_BASE_URL = config["solr"]["base_url"].rstrip("/")
SOLR_VERIFY_SSL = config["solr"].get("verify_ssl", True)
SOLR_TIMEOUT = config["solr"].get("timeout", 30)

SOLR_CORES = {
    "statistics": config["solr"].get("statistics_core", "statistics"),
    "search": config["solr"].get("search_core", "search"),
    "oai": config["solr"].get("oai_core", "oai")
}

print(f"Solr base URL: {SOLR_BASE_URL}")
print(f"Solr cores: {SOLR_CORES}")
print()


def solr_get(path, params=None):
    url = f"{SOLR_BASE_URL}/{path.lstrip('/')}"

    response = requests.get(
        url,
        params=params,
        timeout=SOLR_TIMEOUT,
        verify=SOLR_VERIFY_SSL
    )

    response.raise_for_status()
    return response.json()


def solr_select(core_name, params=None):
    default_params = {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }

    if params:
        default_params.update(params)

    return solr_get(f"{core_name}/select", default_params)


print("=== Solr connection check ===")

for logical_name, core_name in SOLR_CORES.items():
    try:
        response = solr_select(core_name)
        num_found = response["response"]["numFound"]

        print(f"Core '{logical_name}' ({core_name}): reachable")
        print(f"Records found: {num_found}")
        print()

    except Exception as e:
        print(f"Core '{logical_name}' ({core_name}): connection failed - {e}")
        print()

## **General Solr Core Status**

### Available Solr Cores

In [ ]:
# =========================================================
# AVAILABLE SOLR CORES
# =========================================================

# - core_name
# - instance_dir
# - data_dir
# - config
# - schema
# - start_time
# - uptime_ms
# - num_docs
# - max_doc
# - deleted_docs
# - index_size
# - reference_month
# - environment
# - source
# - metric_definition

solr_cores_response = solr_get(
    "admin/cores",
    {
        "action": "STATUS",
        "wt": "json"
    }
)

available_cores = []

for core_name, core_info in solr_cores_response.get("status", {}).items():
    index_info = core_info.get("index", {})

    available_cores.append({
        "core_name": core_name,
        "instance_dir": core_info.get("instanceDir"),
        "data_dir": core_info.get("dataDir"),
        "config": core_info.get("config"),
        "schema": core_info.get("schema"),
        "start_time": core_info.get("startTime"),
        "uptime_ms": core_info.get("uptime"),
        "num_docs": index_info.get("numDocs"),
        "max_doc": index_info.get("maxDoc"),
        "deleted_docs": index_info.get("deletedDocs"),
        "index_size": index_info.get("size")
    })

df_available_solr_cores = pd.DataFrame(available_cores)

df_available_solr_cores["reference_month"] = REPORT_MONTH
df_available_solr_cores["environment"] = ENVIRONMENT
df_available_solr_cores["source"] = "solr"
df_available_solr_cores["metric_definition"] = (
    "available Solr cores with basic status and index information"
)

output_available_solr_cores = export_path("solr_available_cores.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_available_solr_cores.to_csv(output_available_solr_cores, index=False)

print(f"Available Solr cores: {len(df_available_solr_cores)}")

display(df_available_solr_cores)

print(f"File saved to: {output_available_solr_cores.resolve()}")

### Index size by SOLR Core

In [ ]:
# =========================================================
# INDEX SIZE BY SOLR CORE
# =========================================================

# - core_logical_name
# - core_name
# - index_size
# - index_size_bytes
# - num_docs
# - max_doc
# - deleted_docs
# - reference_month
# - environment
# - source
# - metric_definition

solr_cores_status_response = solr_get(
    "admin/cores",
    {
        "action": "STATUS",
        "wt": "json"
    }
)

index_size_rows = []

for core_logical_name, core_name in SOLR_CORES.items():
    core_info = (
        solr_cores_status_response
        .get("status", {})
        .get(core_name, {})
    )

    index_info = core_info.get("index", {})

    index_size_rows.append({
        "core_logical_name": core_logical_name,
        "core_name": core_name,
        "index_size": index_info.get("size"),
        "index_size_bytes": index_info.get("sizeInBytes"),
        "num_docs": index_info.get("numDocs"),
        "max_doc": index_info.get("maxDoc"),
        "deleted_docs": index_info.get("deletedDocs")
    })

df_solr_index_size_by_core = pd.DataFrame(index_size_rows)

df_solr_index_size_by_core["reference_month"] = REPORT_MONTH
df_solr_index_size_by_core["environment"] = ENVIRONMENT
df_solr_index_size_by_core["source"] = "solr"
df_solr_index_size_by_core["metric_definition"] = (
    "index size and basic index statistics for each configured Solr core"
)

output_solr_index_size_by_core = export_path("solr_index_size_by_core.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_solr_index_size_by_core.to_csv(output_solr_index_size_by_core, index=False)

print(f"Solr cores analyzed for index size: {len(df_solr_index_size_by_core)}")

display(df_solr_index_size_by_core)

print(f"File saved to: {output_solr_index_size_by_core.resolve()}")

### Record Count by Solr Core

In [ ]:
# =========================================================
# RECORD COUNT BY SOLR CORE
# =========================================================

# - core_logical_name
# - core_name
# - records_count
# - query
# - reference_month
# - environment
# - source
# - metric_definition

solr_core_record_counts = []

for core_logical_name, core_name in SOLR_CORES.items():
    try:
        response = solr_select(
            core_name,
            {
                "q": "*:*",
                "rows": 0,
                "wt": "json"
            }
        )

        records_count = response.get("response", {}).get("numFound")

        solr_core_record_counts.append({
            "core_logical_name": core_logical_name,
            "core_name": core_name,
            "records_count": records_count,
            "query": "*:*"
        })

    except Exception as e:
        solr_core_record_counts.append({
            "core_logical_name": core_logical_name,
            "core_name": core_name,
            "records_count": None,
            "query": "*:*",
            "error": str(e)
        })

df_solr_core_record_counts = pd.DataFrame(solr_core_record_counts)

df_solr_core_record_counts["reference_month"] = REPORT_MONTH
df_solr_core_record_counts["environment"] = ENVIRONMENT
df_solr_core_record_counts["source"] = "solr"
df_solr_core_record_counts["metric_definition"] = (
    "total number of records indexed in each configured Solr core"
)

output_solr_core_record_counts = export_path("solr_record_count_by_core.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_solr_core_record_counts.to_csv(output_solr_core_record_counts, index=False)

print(f"Solr cores analyzed: {len(df_solr_core_record_counts)}")

display(df_solr_core_record_counts)

print(f"File saved to: {output_solr_core_record_counts.resolve()}")

### Distinct Event Types in the Statistics Core

In [ ]:
# - statistics_type
# - events_count
# - core_name
# - facet_field
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_statistics_type = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "rows": 0,
        "facet": "true",
        "facet.field": "statistics_type",
        "facet.mincount": 1,
        "facet.sort": "count",
        "wt": "json"
    }
)

facet_values = (
    response_statistics_type
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("statistics_type", [])
)

statistics_type_rows = []

for i in range(0, len(facet_values), 2):
    statistics_type_rows.append({
        "statistics_type": facet_values[i],
        "events_count": facet_values[i + 1],
        "core_name": statistics_core,
        "facet_field": "statistics_type"
    })

df_statistics_event_types = pd.DataFrame(statistics_type_rows)

df_statistics_event_types["reference_month"] = REPORT_MONTH
df_statistics_event_types["environment"] = ENVIRONMENT
df_statistics_event_types["source"] = "solr"
df_statistics_event_types["metric_definition"] = (
    "distinct values of the statistics_type field in the Solr statistics core"
)

output_statistics_event_types = export_path("solr_statistics_distinct_event_types.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_statistics_event_types.to_csv(output_statistics_event_types, index=False)

print(f"Distinct event types in the statistics core: {len(df_statistics_event_types)}")

display(df_statistics_event_types)

print(f"File saved to: {output_statistics_event_types.resolve()}")

### Distinct Object Types in the Statistics Core

In [ ]:
# =========================================================
# DISTINCT OBJECT TYPES IN THE STATISTICS CORE
# =========================================================

# - object_type
# - object_type_label
# - events_count
# - core_name
# - facet_field
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

statistics_object_type_legend = {
    "0": "bitstream",
    "2": "item",
    "3": "collection",
    "4": "community"
}

response_statistics_object_type = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "rows": 0,
        "facet": "true",
        "facet.field": "type",
        "facet.mincount": 1,
        "facet.sort": "count",
        "wt": "json"
    }
)

facet_values = (
    response_statistics_object_type
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("type", [])
)

statistics_object_type_rows = []

for i in range(0, len(facet_values), 2):
    object_type = str(facet_values[i])

    statistics_object_type_rows.append({
        "object_type": object_type,
        "object_type_label": statistics_object_type_legend.get(
            object_type,
            "unknown"
        ),
        "events_count": facet_values[i + 1],
        "core_name": statistics_core,
        "facet_field": "type"
    })

df_statistics_object_types = pd.DataFrame(statistics_object_type_rows)

df_statistics_object_types["reference_month"] = REPORT_MONTH
df_statistics_object_types["environment"] = ENVIRONMENT
df_statistics_object_types["source"] = "solr"
df_statistics_object_types["metric_definition"] = (
    "distinct values of the type field in the Solr statistics core; "
    "legend: 0=bitstream, 2=item, 3=collection, 4=community"
)

output_statistics_object_types = export_path("solr_statistics_distinct_object_types.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_statistics_object_types.to_csv(output_statistics_object_types, index=False)

print(f"Distinct object types in the statistics core: {len(df_statistics_object_types)}")
print("Object type legend: 0 = bitstream, 2 = item, 3 = collection, 4 = community")

display(df_statistics_object_types)

print(f"File saved to: {output_statistics_object_types.resolve()}")

### OAI Sets and Available Values

The Solr `oai` core is used to support the OAI-PMH exposure of repository metadata. Unlike the `statistics` core, which records usage events such as item views and bitstream downloads, the `oai` core is related to metadata harvesting and interoperability.

OAI sets are logical groupings of records exposed through OAI-PMH. In a DSpace repository, they may correspond to collections, communities, or locally configured harvesting groups. They allow external services and aggregators to harvest only a specific subset of the repository instead of retrieving all available records.

This analysis checks which OAI-related values are available in the Solr `oai` core and how many records are associated with each value. It is useful for verifying whether the repository metadata is properly exposed, whether the OAI index is populated, and whether the available sets or grouping fields are coherent with the repository structure.

This metric does not measure user activity. It should be interpreted as a diagnostic check of metadata exposure and harvestability.

In [ ]:
# =========================================================
# OAI SETS AND AVAILABLE VALUES
# =========================================================

# - core_name
# - facet_field
# - field_value
# - records_count
# - reference_month
# - environment
# - source
# - metric_definition

oai_core = SOLR_CORES.get("oai", "oai")

# Candidate fields that may contain OAI-relevant grouping/status values.
# The actual fields depend on the DSpace/Solr schema and local configuration.
oai_candidate_facet_fields = [
    "item.collections",
    "item.communities",
    "item.owningCollection",
    "item.deleted",
    "item.withdrawn",
    "item.public",
    "setSpec",
    "sets",
    "set"
]

# Retrieve one sample document to detect available fields in the OAI core.
response_oai_sample = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 1,
        "wt": "json"
    }
)

sample_docs = response_oai_sample.get("response", {}).get("docs", [])

if sample_docs:
    available_oai_fields = set(sample_docs[0].keys())
else:
    available_oai_fields = set()

available_oai_facet_fields = [
    field for field in oai_candidate_facet_fields
    if field in available_oai_fields
]

oai_sets_rows = []

for facet_field in available_oai_facet_fields:
    response_oai_facet = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": 0,
            "facet": "true",
            "facet.field": facet_field,
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    facet_values = (
        response_oai_facet
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get(facet_field, [])
    )

    for i in range(0, len(facet_values), 2):
        oai_sets_rows.append({
            "core_name": oai_core,
            "facet_field": facet_field,
            "field_value": facet_values[i],
            "records_count": facet_values[i + 1]
        })

df_oai_sets_available_values = pd.DataFrame(oai_sets_rows)

df_oai_sets_available_values["reference_month"] = REPORT_MONTH
df_oai_sets_available_values["environment"] = ENVIRONMENT
df_oai_sets_available_values["source"] = "solr"
df_oai_sets_available_values["metric_definition"] = (
    "available OAI-related values detected through faceting on relevant fields "
    "in the Solr OAI core"
)

output_oai_sets_available_values = export_path("solr_oai_sets_available_values.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_oai_sets_available_values.to_csv(output_oai_sets_available_values, index=False)

print(f"OAI facet fields detected: {len(available_oai_facet_fields)}")
print(f"OAI values extracted: {len(df_oai_sets_available_values)}")

print("Detected OAI facet fields:")
display(pd.DataFrame({"facet_field": available_oai_facet_fields}))

display(df_oai_sets_available_values.head())

print(f"File saved to: {output_oai_sets_available_values.resolve()}")

### First 30 events in the statistics core

In [ ]:
# =========================================================
# FIRST 30 EVENTS IN THE STATISTICS CORE
# =========================================================

# - event_id
# - event_time
# - statistics_type
# - object_type
# - object_type_label
# - core_name
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

statistics_object_type_legend = {
    "0": "bitstream",
    "1": "bundle",
    "2": "item",
    "3": "collection",
    "4": "community",
    "5": "site",
    "6": "group",
    "7": "eperson"
}

response_first_30_statistics_events = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "rows": 30,
        "sort": "time asc",
        "fl": "id,time,statistics_type,type",
        "wt": "json"
    }
)

docs = (
    response_first_30_statistics_events
    .get("response", {})
    .get("docs", [])
)

first_30_events_rows = []

for doc in docs:
    object_type = str(doc.get("type"))

    first_30_events_rows.append({
        "event_id": doc.get("id"),
        "event_time": doc.get("time"),
        "statistics_type": doc.get("statistics_type"),
        "object_type": object_type,
        "object_type_label": statistics_object_type_legend.get(
            object_type,
            "unknown"
        ),
        "core_name": statistics_core,
        "query": "*:* SORT time asc ROWS 30"
    })

df_first_30_statistics_events = pd.DataFrame(first_30_events_rows)

df_first_30_statistics_events["reference_month"] = REPORT_MONTH
df_first_30_statistics_events["environment"] = ENVIRONMENT
df_first_30_statistics_events["source"] = "solr"
df_first_30_statistics_events["metric_definition"] = (
    "first 30 events recorded in the Solr statistics core, including event date, "
    "statistics_type and DSpace object type"
)

output_first_30_statistics_events = export_path("solr_first_30_statistics_events.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_first_30_statistics_events.to_csv(output_first_30_statistics_events, index=False)

print(f"First statistics events extracted: {len(df_first_30_statistics_events)}")

if not df_first_30_statistics_events.empty:
    print(
        "First event date: "
        f"{df_first_30_statistics_events['event_time'].min()}"
    )
    print(
        "Last event date among first 30: "
        f"{df_first_30_statistics_events['event_time'].max()}"
    )
else:
    print("No events found in the statistics core.")

display(df_first_30_statistics_events)

print(f"File saved to: {output_first_30_statistics_events.resolve()}")

## **General usage**

### Total item views

In [ ]:
# =========================================================
# TOTAL ITEM VIEWS
# =========================================================

# - metric_name
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - total_events
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_total_item_views = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:2",
            "statistics_type:view",
            "-isBot:true"
        ],
        "rows": 0,
        "wt": "json"
    }
)

total_item_views = (
    response_total_item_views
    .get("response", {})
    .get("numFound", 0)
)

df_total_item_views = pd.DataFrame([
    {
        "metric_name": "total_item_views",
        "core_name": statistics_core,
        "object_type": "2",
        "object_type_label": "item",
        "statistics_type": "view",
        "total_events": total_item_views,
        "bot_filter": "excluded",
        "query": "type:2 AND statistics_type:view AND -isBot:true"
    }
])

df_total_item_views["reference_month"] = REPORT_MONTH
df_total_item_views["environment"] = ENVIRONMENT
df_total_item_views["source"] = "solr"
df_total_item_views["metric_definition"] = (
    "total number of item view events recorded in the Solr statistics core, excluding bot events"
)

output_total_item_views = export_path("solr_total_item_views.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_total_item_views.to_csv(output_total_item_views, index=False)

print(f"Total item views excluding bots: {total_item_views}")

display(df_total_item_views)

print(f"File saved to: {output_total_item_views.resolve()}")

### Total bitstream downloads

In [ ]:
# =========================================================
# TOTAL BITSTREAM DOWNLOADS
# =========================================================

# - metric_name
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - total_events
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_total_bitstream_downloads = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "wt": "json"
    }
)

total_bitstream_downloads = (
    response_total_bitstream_downloads
    .get("response", {})
    .get("numFound", 0)
)

df_total_bitstream_downloads = pd.DataFrame([
    {
        "metric_name": "total_bitstream_downloads",
        "core_name": statistics_core,
        "object_type": "0",
        "object_type_label": "bitstream",
        "statistics_type": "view",
        "bundle_name": "ORIGINAL",
        "total_events": total_bitstream_downloads,
        "bot_filter": "excluded",
        "query": (
            "type:0 AND statistics_type:view "
            "AND bundleName:ORIGINAL AND -isBot:true"
        )
    }
])

df_total_bitstream_downloads["reference_month"] = REPORT_MONTH
df_total_bitstream_downloads["environment"] = ENVIRONMENT
df_total_bitstream_downloads["source"] = "solr"
df_total_bitstream_downloads["metric_definition"] = (
    "total number of bitstream view/download events recorded in the Solr statistics core, "
    "restricted to the ORIGINAL bundle and excluding bot events"
)

output_total_bitstream_downloads = export_path("solr_total_bitstream_downloads.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_total_bitstream_downloads.to_csv(output_total_bitstream_downloads, index=False)

print(f"Total bitstream downloads excluding bots: {total_bitstream_downloads}")

display(df_total_bitstream_downloads)

print(f"File saved to: {output_total_bitstream_downloads.resolve()}")

### Total events

In [ ]:
# =========================================================
# TOTAL EVENTS
# =========================================================

# - metric_name
# - core_name
# - total_events
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_total_events = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_events = (
    response_total_events
    .get("response", {})
    .get("numFound", 0)
)

df_total_events = pd.DataFrame([
    {
        "metric_name": "total_events",
        "core_name": statistics_core,
        "total_events": total_events,
        "bot_filter": "none",
        "query": "*:*"
    }
])

df_total_events["reference_month"] = REPORT_MONTH
df_total_events["environment"] = ENVIRONMENT
df_total_events["source"] = "solr"
df_total_events["metric_definition"] = (
    "total number of events recorded in the Solr statistics core, including bot and non-bot events"
)

output_total_events = export_path("solr_total_events.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_total_events.to_csv(output_total_events, index=False)

print(f"Total events: {total_events}")

display(df_total_events)

print(f"File saved to: {output_total_events.resolve()}")

### Non-bot events

In [ ]:
# =========================================================
# NON-BOT EVENTS
# =========================================================

# - metric_name
# - core_name
# - total_events
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_non_bot_events = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "-isBot:true"
        ],
        "rows": 0,
        "wt": "json"
    }
)

non_bot_events = (
    response_non_bot_events
    .get("response", {})
    .get("numFound", 0)
)

df_non_bot_events = pd.DataFrame([
    {
        "metric_name": "non_bot_events",
        "core_name": statistics_core,
        "total_events": non_bot_events,
        "bot_filter": "excluded",
        "query": "*:* AND -isBot:true"
    }
])

df_non_bot_events["reference_month"] = REPORT_MONTH
df_non_bot_events["environment"] = ENVIRONMENT
df_non_bot_events["source"] = "solr"
df_non_bot_events["metric_definition"] = (
    "total number of events recorded in the Solr statistics core, excluding bot events"
)

output_non_bot_events = export_path("solr_non_bot_events.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_non_bot_events.to_csv(output_non_bot_events, index=False)

print(f"Non-bot events: {non_bot_events}")

display(df_non_bot_events)

print(f"File saved to: {output_non_bot_events.resolve()}")

## **Top Items and Bitstreams**

### Most viewed items

In [ ]:
# =========================================================
# MOST VIEWED ITEMS
# =========================================================

# - item_id
# - views_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bot_filter
# - facet_field
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_most_viewed_items = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:2",
            "statistics_type:view",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": 50,
        "wt": "json"
    }
)

facet_values = (
    response_most_viewed_items
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

most_viewed_items_rows = []

for i in range(0, len(facet_values), 2):
    most_viewed_items_rows.append({
        "item_id": facet_values[i],
        "views_count": facet_values[i + 1],
        "core_name": statistics_core,
        "object_type": "2",
        "object_type_label": "item",
        "statistics_type": "view",
        "bot_filter": "excluded",
        "facet_field": "id",
        "query": (
            "type:2 AND statistics_type:view "
            "AND -isBot:true "
            "FACET id"
        )
    })

df_most_viewed_items = pd.DataFrame(most_viewed_items_rows)

df_most_viewed_items["reference_month"] = REPORT_MONTH
df_most_viewed_items["environment"] = ENVIRONMENT
df_most_viewed_items["source"] = "solr"
df_most_viewed_items["metric_definition"] = (
    "most viewed items based on item view events recorded in the Solr statistics core, "
    "excluding bot events"
)

output_most_viewed_items = export_dir / "solr_most_viewed_items.csv"

export_dir.mkdir(parents=True, exist_ok=True)
df_most_viewed_items.to_csv(output_most_viewed_items, index=False)

print(f"Most viewed items extracted: {len(df_most_viewed_items)}")

display(df_most_viewed_items.head())

print(f"File saved to: {output_most_viewed_items.resolve()}")

### Most downloaded items

In [ ]:
# =========================================================
# MOST DOWNLOADED ITEMS
# =========================================================

# - item_id
# - downloads_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_most_downloaded_items = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": 50,
        "wt": "json"
    }
)

facet_values = (
    response_most_downloaded_items
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

most_downloaded_items_rows = []

for i in range(0, len(facet_values), 2):
    most_downloaded_items_rows.append({
        "item_id": facet_values[i],
        "downloads_count": facet_values[i + 1],
        "core_name": statistics_core,
        "object_type": "0",
        "object_type_label": "bitstream",
        "statistics_type": "view",
        "bundle_name": "ORIGINAL",
        "bot_filter": "excluded",
        "query": (
            "type:0 AND statistics_type:view "
            "AND bundleName:ORIGINAL AND -isBot:true "
            "FACET owningItem"
        )
    })

df_most_downloaded_items = pd.DataFrame(most_downloaded_items_rows)

df_most_downloaded_items["reference_month"] = REPORT_MONTH
df_most_downloaded_items["environment"] = ENVIRONMENT
df_most_downloaded_items["source"] = "solr"
df_most_downloaded_items["metric_definition"] = (
    "most downloaded items based on bitstream view/download events aggregated by owningItem, "
    "restricted to the ORIGINAL bundle and excluding bot events"
)

output_most_downloaded_items = export_path("solr_most_downloaded_items.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_most_downloaded_items.to_csv(output_most_downloaded_items, index=False)

print(f"Most downloaded items extracted: {len(df_most_downloaded_items)}")

display(df_most_downloaded_items.head())

print(f"File saved to: {output_most_downloaded_items.resolve()}")

### Most downloaded bitstreams

In [ ]:
# =========================================================
# MOST DOWNLOADED BITSTREAMS
# =========================================================

# - bitstream_id
# - bitstream_name
# - downloads_count
# - owning_item_id
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

# Candidate filename fields. The actual field depends on the local Solr schema.
candidate_filename_fields = [
    "name",
    "filename",
    "fileName",
    "bitstreamName",
    "dc.title",
    "title"
]

response_sample_bitstream_event = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view"
        ],
        "rows": 1,
        "wt": "json"
    }
)

sample_docs = (
    response_sample_bitstream_event
    .get("response", {})
    .get("docs", [])
)

available_fields = set(sample_docs[0].keys()) if sample_docs else set()

filename_field = None

for field in candidate_filename_fields:
    if field in available_fields:
        filename_field = field
        break

field_list = [
    "id",
    "owningItem"
]

if filename_field is not None:
    field_list.append(filename_field)

response_most_downloaded_bitstreams = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": 50,
        "wt": "json"
    }
)

facet_values = (
    response_most_downloaded_bitstreams
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

most_downloaded_bitstreams_rows = []

for i in range(0, len(facet_values), 2):
    bitstream_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_bitstream_doc = solr_select(
        statistics_core,
        {
            "q": f'id:"{bitstream_id}"',
            "fq": [
                "type:0",
                "statistics_type:view"
            ],
            "rows": 1,
            "fl": ",".join(field_list),
            "wt": "json"
        }
    )

    docs = (
        response_bitstream_doc
        .get("response", {})
        .get("docs", [])
    )

    if docs:
        doc = docs[0]
        owning_item_id = doc.get("owningItem")
        bitstream_name = doc.get(filename_field) if filename_field else None
    else:
        owning_item_id = None
        bitstream_name = None

    most_downloaded_bitstreams_rows.append({
        "bitstream_id": bitstream_id,
        "bitstream_name": bitstream_name,
        "downloads_count": downloads_count,
        "owning_item_id": owning_item_id,
        "core_name": statistics_core,
        "object_type": "0",
        "object_type_label": "bitstream",
        "statistics_type": "view",
        "bundle_name": "ORIGINAL",
        "bot_filter": "excluded",
        "query": (
            "type:0 AND statistics_type:view "
            "AND bundleName:ORIGINAL AND -isBot:true "
            "FACET id"
        )
    })

df_most_downloaded_bitstreams = pd.DataFrame(most_downloaded_bitstreams_rows)

df_most_downloaded_bitstreams["reference_month"] = REPORT_MONTH
df_most_downloaded_bitstreams["environment"] = ENVIRONMENT
df_most_downloaded_bitstreams["source"] = "solr"
df_most_downloaded_bitstreams["metric_definition"] = (
    "most downloaded bitstreams based on bitstream view/download events, "
    "restricted to the ORIGINAL bundle and excluding bot events; "
    "bitstream filename is included only if available in the Solr statistics core"
)

output_most_downloaded_bitstreams = export_path("solr_most_downloaded_bitstreams.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_most_downloaded_bitstreams.to_csv(output_most_downloaded_bitstreams, index=False)

if filename_field is not None:
    print(f"Filename field detected in Solr statistics core: {filename_field}")
else:
    print("No filename field detected in Solr statistics core. Use PostgreSQL join to retrieve filenames.")

print(f"Most downloaded bitstreams extracted: {len(df_most_downloaded_bitstreams)}")

display(df_most_downloaded_bitstreams.head())

print(f"File saved to: {output_most_downloaded_bitstreams.resolve()}")

### Top viewed items in the reference month

In [ ]:
# =========================================================
# TOP ITEMS IN THE REFERENCE MONTH
# =========================================================

# - item_id
# - views_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

response_top_items_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:2",
            "statistics_type:view",
            "-isBot:true",
            f"time:[{period_start_solr} TO {period_end_solr}}}"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": 50,
        "wt": "json"
    }
)

facet_values = (
    response_top_items_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

top_items_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    top_items_reference_month_rows.append({
        "item_id": facet_values[i],
        "views_count": facet_values[i + 1],
        "core_name": statistics_core,
        "object_type": "2",
        "object_type_label": "item",
        "statistics_type": "view",
        "bot_filter": "excluded",
        "period_start": reference_month_start.strftime("%Y-%m-%d"),
        "period_end": reference_month_end.strftime("%Y-%m-%d"),
        "query": (
            "type:2 AND statistics_type:view AND -isBot:true "
            f"AND time:[{period_start_solr} TO {period_end_solr}}} "
            "FACET id"
        )
    })

df_top_items_reference_month = pd.DataFrame(top_items_reference_month_rows)

df_top_items_reference_month["reference_month"] = REPORT_MONTH
df_top_items_reference_month["environment"] = ENVIRONMENT
df_top_items_reference_month["source"] = "solr"
df_top_items_reference_month["metric_definition"] = (
    "top viewed items in the reference month based on item view events recorded "
    "in the Solr statistics core, excluding bot events"
)

output_top_items_reference_month = export_path("solr_top_items_reference_month.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_top_items_reference_month.to_csv(output_top_items_reference_month, index=False)

print(f"Top items in the reference month extracted: {len(df_top_items_reference_month)}")
print(f"Reference period: {reference_month_start.strftime('%Y-%m-%d')} to {reference_month_end.strftime('%Y-%m-%d')}")

display(df_top_items_reference_month.head())

print(f"File saved to: {output_top_items_reference_month.resolve()}")

### Top downloaded items in the reference month

In [ ]:
# =========================================================
# TOP DOWNLOADED ITEMS IN THE REFERENCE MONTH
# =========================================================

# - item_id
# - bitstream_ids
# - downloads_count
# - bitstreams_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

response_top_downloaded_items_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": 50,
        "wt": "json"
    }
)

facet_values = (
    response_top_downloaded_items_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

top_downloaded_items_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                time_filter,
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams_reference_month
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    top_downloaded_items_reference_month_rows.append({
        "item_id": item_id,
        "bitstream_ids": "; ".join(bitstream_ids),
        "downloads_count": downloads_count,
        "bitstreams_count": len(bitstream_ids),
        "core_name": statistics_core,
        "object_type": "0",
        "object_type_label": "bitstream",
        "statistics_type": "view",
        "bundle_name": "ORIGINAL",
        "bot_filter": "excluded",
        "period_start": reference_month_start.strftime("%Y-%m-%d"),
        "period_end": reference_month_end.strftime("%Y-%m-%d"),
        "query": (
            "type:0 AND statistics_type:view "
            "AND bundleName:ORIGINAL AND -isBot:true "
            f"AND {time_filter} "
            "FACET owningItem; nested FACET id by owningItem"
        )
    })

df_top_downloaded_items_reference_month = pd.DataFrame(
    top_downloaded_items_reference_month_rows
)

df_top_downloaded_items_reference_month["reference_month"] = REPORT_MONTH
df_top_downloaded_items_reference_month["environment"] = ENVIRONMENT
df_top_downloaded_items_reference_month["source"] = "solr"
df_top_downloaded_items_reference_month["metric_definition"] = (
    "top downloaded items in the reference month based on bitstream view/download "
    "events aggregated by owningItem, including associated downloaded bitstream ids; "
    "restricted to the ORIGINAL bundle and excluding bot events"
)

output_top_downloaded_items_reference_month = (
    export_path("solr_top_downloaded_items_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_top_downloaded_items_reference_month.to_csv(
    output_top_downloaded_items_reference_month,
    index=False
)

print(
    "Top downloaded items in the reference month extracted: "
    f"{len(df_top_downloaded_items_reference_month)}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_top_downloaded_items_reference_month.head())

print(f"File saved to: {output_top_downloaded_items_reference_month.resolve()}")

## **Referrers and Access**

### Top referrers

In [ ]:
# =========================================================
# TOP REFERRERS BY BOT STATUS
# =========================================================

# - referrer
# - referrer_normalized
# - bot_status
# - events_count
# - core_name
# - statistics_type
# - facet_field
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_top_referrers_by_bot_status = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:view"
        ],
        "rows": 0,
        "facet": "true",
        "facet.pivot": "referrer,isBot",
        "facet.mincount": 1,
        "facet.limit": 50,
        "wt": "json"
    }
)

pivot_values = (
    response_top_referrers_by_bot_status
    .get("facet_counts", {})
    .get("facet_pivot", {})
    .get("referrer,isBot", [])
)

top_referrers_by_bot_status_rows = []

for referrer_entry in pivot_values:
    referrer = referrer_entry.get("value")
    referrer_count = referrer_entry.get("count", 0)

    if referrer is None or str(referrer).strip() == "":
        referrer_normalized = "direct_or_unknown"
    else:
        referrer_normalized = str(referrer)

    bot_pivot = referrer_entry.get("pivot", [])

    if bot_pivot:
        for bot_entry in bot_pivot:
            is_bot_value = bot_entry.get("value")
            events_count = bot_entry.get("count", 0)

            if is_bot_value is True or str(is_bot_value).lower() == "true":
                bot_status = "bot"
            elif is_bot_value is False or str(is_bot_value).lower() == "false":
                bot_status = "non_bot"
            else:
                bot_status = "unknown"

            top_referrers_by_bot_status_rows.append({
                "referrer": referrer,
                "referrer_normalized": referrer_normalized,
                "bot_status": bot_status,
                "events_count": events_count,
                "core_name": statistics_core,
                "statistics_type": "view",
                "facet_field": "referrer,isBot",
                "query": "statistics_type:view FACET PIVOT referrer,isBot"
            })

    else:
        top_referrers_by_bot_status_rows.append({
            "referrer": referrer,
            "referrer_normalized": referrer_normalized,
            "bot_status": "unknown",
            "events_count": referrer_count,
            "core_name": statistics_core,
            "statistics_type": "view",
            "facet_field": "referrer,isBot",
            "query": "statistics_type:view FACET PIVOT referrer,isBot"
        })

df_top_referrers_by_bot_status = pd.DataFrame(
    top_referrers_by_bot_status_rows
)

df_top_referrers_by_bot_status["reference_month"] = REPORT_MONTH
df_top_referrers_by_bot_status["environment"] = ENVIRONMENT
df_top_referrers_by_bot_status["source"] = "solr"
df_top_referrers_by_bot_status["metric_definition"] = (
    "top referrers for view events recorded in the Solr statistics core, "
    "grouped by bot status using the isBot field"
)

output_top_referrers_by_bot_status = (
    export_path("solr_top_referrers_by_bot_status.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_top_referrers_by_bot_status.to_csv(
    output_top_referrers_by_bot_status,
    index=False
)

print(f"Top referrer / bot-status rows extracted: {len(df_top_referrers_by_bot_status)}")

print("Distribution by bot status:")
display(
    df_top_referrers_by_bot_status
    .groupby("bot_status", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print("Top referrers by total events:")
display(
    df_top_referrers_by_bot_status
    .groupby("referrer_normalized", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
    .head(20)
)

display(df_top_referrers_by_bot_status.head())

print(f"File saved to: {output_top_referrers_by_bot_status.resolve()}")

### Top referrers in the reference month

In [ ]:
# =========================================================
# TOP REFERRERS IN THE REFERENCE MONTH
# =========================================================

# - referrer
# - referrer_normalized
# - events_count
# - core_name
# - statistics_type
# - bot_filter
# - facet_field
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

response_top_referrers_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:view",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "referrer",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": 50,
        "wt": "json"
    }
)

facet_values = (
    response_top_referrers_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("referrer", [])
)

top_referrers_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    referrer = facet_values[i]

    if referrer is None or str(referrer).strip() == "":
        referrer_normalized = "direct_or_unknown"
    else:
        referrer_normalized = str(referrer)

    top_referrers_reference_month_rows.append({
        "referrer": referrer,
        "referrer_normalized": referrer_normalized,
        "events_count": facet_values[i + 1],
        "core_name": statistics_core,
        "statistics_type": "view",
        "bot_filter": "excluded",
        "facet_field": "referrer",
        "period_start": reference_month_start.strftime("%Y-%m-%d"),
        "period_end": reference_month_end.strftime("%Y-%m-%d"),
        "query": (
            "statistics_type:view AND -isBot:true "
            f"AND {time_filter} FACET referrer"
        )
    })

df_top_referrers_reference_month = pd.DataFrame(
    top_referrers_reference_month_rows
)

df_top_referrers_reference_month["reference_month"] = REPORT_MONTH
df_top_referrers_reference_month["environment"] = ENVIRONMENT
df_top_referrers_reference_month["source"] = "solr"
df_top_referrers_reference_month["metric_definition"] = (
    "top referrers for view events recorded in the Solr statistics core "
    "during the reference month, excluding bot events; empty referrers are "
    "normalized as direct_or_unknown"
)

output_top_referrers_reference_month = (
    export_path("solr_top_referrers_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_top_referrers_reference_month.to_csv(
    output_top_referrers_reference_month,
    index=False
)

print(
    "Top referrers in the reference month extracted: "
    f"{len(df_top_referrers_reference_month)}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_top_referrers_reference_month.head())

print(f"File saved to: {output_top_referrers_reference_month.resolve()}")

### Access from search engines

In [ ]:
# =========================================================
# ACCESSES FROM SEARCH ENGINES
# =========================================================

# - search_engine
# - referrer_domain
# - events_count
# - core_name
# - statistics_type
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse

statistics_core = SOLR_CORES.get("statistics", "statistics")

search_engine_domains = {
    "google": ["google."],
    "bing": ["bing.com"],
    "duckduckgo": ["duckduckgo.com"],
    "yahoo": ["yahoo."],
    "yandex": ["yandex."],
    "baidu": ["baidu."],
    "ecosia": ["ecosia.org"],
    "qwant": ["qwant.com"],
    "startpage": ["startpage.com"]
}


def classify_search_engine(referrer):
    if referrer is None or str(referrer).strip() == "":
        return None

    referrer_lower = str(referrer).lower()

    for engine_name, domains in search_engine_domains.items():
        if any(domain in referrer_lower for domain in domains):
            return engine_name

    return None


def extract_domain(referrer):
    if referrer is None or str(referrer).strip() == "":
        return None

    try:
        return urlparse(str(referrer)).netloc.lower()
    except Exception:
        return None


response_search_engine_referrers = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:view",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "referrer",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_search_engine_referrers
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("referrer", [])
)

search_engine_rows = []

for i in range(0, len(facet_values), 2):
    referrer = facet_values[i]
    events_count = facet_values[i + 1]

    search_engine = classify_search_engine(referrer)

    if search_engine is not None:
        search_engine_rows.append({
            "search_engine": search_engine,
            "referrer_domain": extract_domain(referrer),
            "events_count": events_count,
            "core_name": statistics_core,
            "statistics_type": "view",
            "bot_filter": "excluded",
            "query": (
                "statistics_type:view AND -isBot:true "
                "FACET referrer; filtered in Python by search engine domains"
            )
        })

df_accesses_from_search_engines = pd.DataFrame(search_engine_rows)

if not df_accesses_from_search_engines.empty:
    df_accesses_from_search_engines = (
        df_accesses_from_search_engines
        .groupby(
            [
                "search_engine",
                "referrer_domain",
                "core_name",
                "statistics_type",
                "bot_filter",
                "query"
            ],
            dropna=False
        )["events_count"]
        .sum()
        .reset_index()
        .sort_values("events_count", ascending=False)
    )

df_accesses_from_search_engines["reference_month"] = REPORT_MONTH
df_accesses_from_search_engines["environment"] = ENVIRONMENT
df_accesses_from_search_engines["source"] = "solr"
df_accesses_from_search_engines["metric_definition"] = (
    "view events whose referrer matches known search engine domains, excluding bot events"
)

output_accesses_from_search_engines = export_path("solr_accesses_from_search_engines.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_accesses_from_search_engines.to_csv(output_accesses_from_search_engines, index=False)

print(
    "Accesses from search engines excluding bots: "
    f"{df_accesses_from_search_engines['events_count'].sum() if not df_accesses_from_search_engines.empty else 0}"
)

display(df_accesses_from_search_engines.head())

print(f"File saved to: {output_accesses_from_search_engines.resolve()}")

### Access from search engines in the reference month

In [ ]:
# =========================================================
# ACCESSES FROM SEARCH ENGINES IN THE REFERENCE MONTH
# =========================================================

# - search_engine
# - referrer_domain
# - events_count
# - core_name
# - statistics_type
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

search_engine_domains = {
    "google": ["google."],
    "bing": ["bing.com"],
    "duckduckgo": ["duckduckgo.com"],
    "yahoo": ["yahoo."],
    "yandex": ["yandex."],
    "baidu": ["baidu."],
    "ecosia": ["ecosia.org"],
    "qwant": ["qwant.com"],
    "startpage": ["startpage.com"]
}


def classify_search_engine(referrer):
    if referrer is None or str(referrer).strip() == "":
        return None

    referrer_lower = str(referrer).lower()

    for engine_name, domains in search_engine_domains.items():
        if any(domain in referrer_lower for domain in domains):
            return engine_name

    return None


def extract_domain(referrer):
    if referrer is None or str(referrer).strip() == "":
        return None

    try:
        return urlparse(str(referrer)).netloc.lower()
    except Exception:
        return None


response_search_engine_referrers_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:view",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "referrer",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_search_engine_referrers_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("referrer", [])
)

search_engine_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    referrer = facet_values[i]
    events_count = facet_values[i + 1]

    search_engine = classify_search_engine(referrer)

    if search_engine is not None:
        search_engine_reference_month_rows.append({
            "search_engine": search_engine,
            "referrer_domain": extract_domain(referrer),
            "events_count": events_count,
            "core_name": statistics_core,
            "statistics_type": "view",
            "bot_filter": "excluded",
            "period_start": reference_month_start.strftime("%Y-%m-%d"),
            "period_end": reference_month_end.strftime("%Y-%m-%d"),
            "query": (
                "statistics_type:view AND -isBot:true "
                f"AND {time_filter} "
                "FACET referrer; filtered in Python by search engine domains"
            )
        })

df_accesses_from_search_engines_reference_month = pd.DataFrame(
    search_engine_reference_month_rows
)

if not df_accesses_from_search_engines_reference_month.empty:
    df_accesses_from_search_engines_reference_month = (
        df_accesses_from_search_engines_reference_month
        .groupby(
            [
                "search_engine",
                "referrer_domain",
                "core_name",
                "statistics_type",
                "bot_filter",
                "period_start",
                "period_end",
                "query"
            ],
            dropna=False
        )["events_count"]
        .sum()
        .reset_index()
        .sort_values("events_count", ascending=False)
    )

df_accesses_from_search_engines_reference_month["reference_month"] = REPORT_MONTH
df_accesses_from_search_engines_reference_month["environment"] = ENVIRONMENT
df_accesses_from_search_engines_reference_month["source"] = "solr"
df_accesses_from_search_engines_reference_month["metric_definition"] = (
    "view events in the reference month whose referrer matches known search engine domains, "
    "excluding bot events"
)

output_accesses_from_search_engines_reference_month = (
    export_path("solr_accesses_from_search_engines_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_accesses_from_search_engines_reference_month.to_csv(
    output_accesses_from_search_engines_reference_month,
    index=False
)

print(
    "Accesses from search engines in the reference month excluding bots: "
    f"{df_accesses_from_search_engines_reference_month['events_count'].sum() if not df_accesses_from_search_engines_reference_month.empty else 0}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_accesses_from_search_engines_reference_month.head())

print(f"File saved to: {output_accesses_from_search_engines_reference_month.resolve()}")

### Most frequent user agents

In [ ]:
# =========================================================
# MOST FREQUENT USER AGENTS BY BOT STATUS
# =========================================================

# - user_agent
# - bot_status
# - events_count
# - core_name
# - statistics_type
# - bot_filter
# - facet_field
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_most_frequent_user_agents = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:view"
        ],
        "rows": 0,
        "facet": "true",
        "facet.pivot": "userAgent,isBot",
        "facet.mincount": 1,
        "facet.limit": 50,
        "wt": "json"
    }
)

pivot_values = (
    response_most_frequent_user_agents
    .get("facet_counts", {})
    .get("facet_pivot", {})
    .get("userAgent,isBot", [])
)

most_frequent_user_agents_rows = []

for user_agent_entry in pivot_values:
    user_agent = user_agent_entry.get("value")
    user_agent_total_count = user_agent_entry.get("count", 0)
    bot_pivot = user_agent_entry.get("pivot", [])

    if bot_pivot:
        for bot_entry in bot_pivot:
            is_bot_value = bot_entry.get("value")
            events_count = bot_entry.get("count", 0)

            if is_bot_value is True or str(is_bot_value).lower() == "true":
                bot_status = "bot"
            elif is_bot_value is False or str(is_bot_value).lower() == "false":
                bot_status = "non_bot"
            else:
                bot_status = "unknown"

            most_frequent_user_agents_rows.append({
                "user_agent": user_agent,
                "bot_status": bot_status,
                "events_count": events_count,
                "core_name": statistics_core,
                "statistics_type": "view",
                "bot_filter": "none",
                "facet_field": "userAgent,isBot",
                "query": "statistics_type:view FACET PIVOT userAgent,isBot"
            })

    else:
        most_frequent_user_agents_rows.append({
            "user_agent": user_agent,
            "bot_status": "unknown",
            "events_count": user_agent_total_count,
            "core_name": statistics_core,
            "statistics_type": "view",
            "bot_filter": "none",
            "facet_field": "userAgent,isBot",
            "query": "statistics_type:view FACET PIVOT userAgent,isBot"
        })

most_frequent_user_agents_columns = [
    "user_agent",
    "bot_status",
    "events_count",
    "core_name",
    "statistics_type",
    "bot_filter",
    "facet_field",
    "query",
]

df_most_frequent_user_agents = pd.DataFrame(
    most_frequent_user_agents_rows,
    columns=most_frequent_user_agents_columns
)

df_most_frequent_user_agents["reference_month"] = REPORT_MONTH
df_most_frequent_user_agents["environment"] = ENVIRONMENT
df_most_frequent_user_agents["source"] = "solr"
df_most_frequent_user_agents["metric_definition"] = (
    "most frequent user agents for view events recorded in the Solr statistics core, "
    "grouped by bot status using the isBot field"
)

output_most_frequent_user_agents = (
    export_path("solr_most_frequent_user_agents_by_bot_status.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_most_frequent_user_agents.to_csv(output_most_frequent_user_agents, index=False)

print(f"Most frequent user agent / bot-status rows extracted: {len(df_most_frequent_user_agents)}")

print("Distribution by bot status:")
display(
    df_most_frequent_user_agents
    .groupby("bot_status", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

display(df_most_frequent_user_agents.head())

print(f"File saved to: {output_most_frequent_user_agents.resolve()}")

### Most frequent user agents in the reference month

In [ ]:
# =========================================================
# MOST FREQUENT USER AGENTS IN THE REFERENCE MONTH BY BOT STATUS
# =========================================================

# - user_agent
# - bot_status
# - events_count
# - core_name
# - statistics_type
# - bot_filter
# - facet_field
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

response_most_frequent_user_agents_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:view",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.pivot": "userAgent,isBot",
        "facet.mincount": 1,
        "facet.limit": 50,
        "wt": "json"
    }
)

pivot_values = (
    response_most_frequent_user_agents_reference_month
    .get("facet_counts", {})
    .get("facet_pivot", {})
    .get("userAgent,isBot", [])
)

most_frequent_user_agents_reference_month_rows = []

for user_agent_entry in pivot_values:
    user_agent = user_agent_entry.get("value")
    user_agent_total_count = user_agent_entry.get("count", 0)
    bot_pivot = user_agent_entry.get("pivot", [])

    if bot_pivot:
        for bot_entry in bot_pivot:
            is_bot_value = bot_entry.get("value")
            events_count = bot_entry.get("count", 0)

            if is_bot_value is True or str(is_bot_value).lower() == "true":
                bot_status = "bot"
            elif is_bot_value is False or str(is_bot_value).lower() == "false":
                bot_status = "non_bot"
            else:
                bot_status = "unknown"

            most_frequent_user_agents_reference_month_rows.append({
                "user_agent": user_agent,
                "bot_status": bot_status,
                "events_count": events_count,
                "core_name": statistics_core,
                "statistics_type": "view",
                "bot_filter": "none",
                "facet_field": "userAgent,isBot",
                "period_start": reference_month_start.strftime("%Y-%m-%d"),
                "period_end": reference_month_end.strftime("%Y-%m-%d"),
                "query": (
                    "statistics_type:view "
                    f"AND {time_filter} "
                    "FACET PIVOT userAgent,isBot"
                )
            })

    else:
        most_frequent_user_agents_reference_month_rows.append({
            "user_agent": user_agent,
            "bot_status": "unknown",
            "events_count": user_agent_total_count,
            "core_name": statistics_core,
            "statistics_type": "view",
            "bot_filter": "none",
            "facet_field": "userAgent,isBot",
            "period_start": reference_month_start.strftime("%Y-%m-%d"),
            "period_end": reference_month_end.strftime("%Y-%m-%d"),
            "query": (
                "statistics_type:view "
                f"AND {time_filter} "
                "FACET PIVOT userAgent,isBot"
            )
        })

most_frequent_user_agents_reference_month_columns = [
    "user_agent",
    "bot_status",
    "events_count",
    "core_name",
    "statistics_type",
    "bot_filter",
    "facet_field",
    "period_start",
    "period_end",
    "query",
]

df_most_frequent_user_agents_reference_month = pd.DataFrame(
    most_frequent_user_agents_reference_month_rows,
    columns=most_frequent_user_agents_reference_month_columns
)

df_most_frequent_user_agents_reference_month["reference_month"] = REPORT_MONTH
df_most_frequent_user_agents_reference_month["environment"] = ENVIRONMENT
df_most_frequent_user_agents_reference_month["source"] = "solr"
df_most_frequent_user_agents_reference_month["metric_definition"] = (
    "most frequent user agents for view events recorded in the Solr statistics core "
    "during the reference month, grouped by bot status using the isBot field"
)

output_most_frequent_user_agents_reference_month = (
    export_path("solr_most_frequent_user_agents_reference_month_by_bot_status.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_most_frequent_user_agents_reference_month.to_csv(
    output_most_frequent_user_agents_reference_month,
    index=False
)

print(
    "Most frequent user agent / bot-status rows in the reference month extracted: "
    f"{len(df_most_frequent_user_agents_reference_month)}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

print("Distribution by bot status:")
display(
    df_most_frequent_user_agents_reference_month
    .groupby("bot_status", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

display(df_most_frequent_user_agents_reference_month.head())

print(f"File saved to: {output_most_frequent_user_agents_reference_month.resolve()}")

### Most frequent IPs/DNS values

In [ ]:
# =========================================================
# MOST FREQUENT IPs / DNS VALUES
# =========================================================

# - network_identifier
# - identifier_type
# - events_count
# - core_name
# - statistics_type
# - bot_filter
# - facet_field
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

candidate_network_fields = [
    "dns",
    "ip",
    "ipAddress",
    "ip_address"
]

response_statistics_sample = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "rows": 1,
        "wt": "json"
    }
)

sample_docs = (
    response_statistics_sample
    .get("response", {})
    .get("docs", [])
)

available_statistics_fields = set(sample_docs[0].keys()) if sample_docs else set()

network_field = None

for field in candidate_network_fields:
    if field in available_statistics_fields:
        network_field = field
        break

most_frequent_network_rows = []

if network_field is not None:
    response_most_frequent_network_values = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "statistics_type:view"
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": network_field,
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": 50,
            "wt": "json"
        }
    )

    facet_values = (
        response_most_frequent_network_values
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get(network_field, [])
    )

    for i in range(0, len(facet_values), 2):
        most_frequent_network_rows.append({
            "network_identifier": facet_values[i],
            "identifier_type": network_field,
            "events_count": facet_values[i + 1],
            "core_name": statistics_core,
            "statistics_type": "view",
            "bot_filter": "none",
            "facet_field": network_field,
            "query": f"statistics_type:view FACET {network_field}"
        })

else:
    most_frequent_network_rows.append({
        "network_identifier": None,
        "identifier_type": "not_available",
        "events_count": None,
        "core_name": statistics_core,
        "statistics_type": "view",
        "bot_filter": "none",
        "facet_field": None,
        "query": "no IP/DNS field available in statistics core"
    })

df_most_frequent_ips_dns = pd.DataFrame(most_frequent_network_rows)

df_most_frequent_ips_dns["reference_month"] = REPORT_MONTH
df_most_frequent_ips_dns["environment"] = ENVIRONMENT
df_most_frequent_ips_dns["source"] = "solr"
df_most_frequent_ips_dns["metric_definition"] = (
    "most frequent IP/DNS values for view events recorded in the Solr statistics core, "
    "including both bot and non-bot events; the available network identifier field is detected automatically"
)

output_most_frequent_ips_dns = export_path("solr_most_frequent_ips_dns.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_most_frequent_ips_dns.to_csv(output_most_frequent_ips_dns, index=False)

if network_field is not None:
    print(f"Network identifier field detected: {network_field}")
    print(f"Most frequent IP/DNS values extracted: {len(df_most_frequent_ips_dns)}")
else:
    print("No IP/DNS field detected in the statistics core.")

display(df_most_frequent_ips_dns.head())

print(f"File saved to: {output_most_frequent_ips_dns.resolve()}")

### Most frequent IPs/DNS values in the reference month

In [ ]:
# =========================================================
# MOST FREQUENT IPs / DNS VALUES IN THE REFERENCE MONTH
# =========================================================

# - network_identifier
# - identifier_type
# - events_count
# - core_name
# - statistics_type
# - bot_filter
# - facet_field
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

candidate_network_fields = [
    "dns",
    "ip",
    "ipAddress",
    "ip_address"
]

response_statistics_sample = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "rows": 1,
        "wt": "json"
    }
)

sample_docs = (
    response_statistics_sample
    .get("response", {})
    .get("docs", [])
)

available_statistics_fields = set(sample_docs[0].keys()) if sample_docs else set()

network_field = None

for field in candidate_network_fields:
    if field in available_statistics_fields:
        network_field = field
        break

most_frequent_network_reference_month_rows = []

if network_field is not None:
    response_most_frequent_network_values_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "statistics_type:view",
                time_filter
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": network_field,
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": 50,
            "wt": "json"
        }
    )

    facet_values = (
        response_most_frequent_network_values_reference_month
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get(network_field, [])
    )

    for i in range(0, len(facet_values), 2):
        most_frequent_network_reference_month_rows.append({
            "network_identifier": facet_values[i],
            "identifier_type": network_field,
            "events_count": facet_values[i + 1],
            "core_name": statistics_core,
            "statistics_type": "view",
            "bot_filter": "none",
            "facet_field": network_field,
            "period_start": reference_month_start.strftime("%Y-%m-%d"),
            "period_end": reference_month_end.strftime("%Y-%m-%d"),
            "query": (
                "statistics_type:view "
                f"AND {time_filter} "
                f"FACET {network_field}"
            )
        })

else:
    most_frequent_network_reference_month_rows.append({
        "network_identifier": None,
        "identifier_type": "not_available",
        "events_count": None,
        "core_name": statistics_core,
        "statistics_type": "view",
        "bot_filter": "none",
        "facet_field": None,
        "period_start": reference_month_start.strftime("%Y-%m-%d"),
        "period_end": reference_month_end.strftime("%Y-%m-%d"),
        "query": "no IP/DNS field available in statistics core"
    })

df_most_frequent_ips_dns_reference_month = pd.DataFrame(
    most_frequent_network_reference_month_rows
)

df_most_frequent_ips_dns_reference_month["reference_month"] = REPORT_MONTH
df_most_frequent_ips_dns_reference_month["environment"] = ENVIRONMENT
df_most_frequent_ips_dns_reference_month["source"] = "solr"
df_most_frequent_ips_dns_reference_month["metric_definition"] = (
    "most frequent IP/DNS values for view events recorded in the Solr statistics core "
    "during the reference month, including both bot and non-bot events; "
    "the available network identifier field is detected automatically"
)

output_most_frequent_ips_dns_reference_month = (
    export_path("solr_most_frequent_ips_dns_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_most_frequent_ips_dns_reference_month.to_csv(
    output_most_frequent_ips_dns_reference_month,
    index=False
)

if network_field is not None:
    print(f"Network identifier field detected: {network_field}")
    print(
        "Most frequent IP/DNS values in the reference month extracted: "
        f"{len(df_most_frequent_ips_dns_reference_month)}"
    )
else:
    print("No IP/DNS field detected in the statistics core.")

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_most_frequent_ips_dns_reference_month.head())

print(f"File saved to: {output_most_frequent_ips_dns_reference_month.resolve()}")

## **Bots and internal events**

### Bot events

In [ ]:
# =========================================================
# BOT EVENTS BY EVENT AND OBJECT TYPE
# =========================================================

# - statistics_type
# - object_type
# - object_type_label
# - bot_status
# - events_count
# - core_name
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

statistics_object_type_legend = {
    "0": "bitstream",
    "1": "bundle",
    "2": "item",
    "3": "collection",
    "4": "community",
    "5": "site",
    "6": "group",
    "7": "eperson"
}

response_bot_events_by_type = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.pivot": "statistics_type,type",
        "facet.mincount": 1,
        "facet.limit": -1,
        "wt": "json"
    }
)

pivot_values = (
    response_bot_events_by_type
    .get("facet_counts", {})
    .get("facet_pivot", {})
    .get("statistics_type,type", [])
)

bot_events_by_type_rows = []

for statistics_type_entry in pivot_values:
    statistics_type = statistics_type_entry.get("value")
    object_type_pivot = statistics_type_entry.get("pivot", [])

    if object_type_pivot:
        for object_type_entry in object_type_pivot:
            object_type = str(object_type_entry.get("value"))
            events_count = object_type_entry.get("count", 0)

            bot_events_by_type_rows.append({
                "statistics_type": statistics_type,
                "object_type": object_type,
                "object_type_label": statistics_object_type_legend.get(
                    object_type,
                    "unknown"
                ),
                "bot_status": "bot",
                "events_count": events_count,
                "core_name": statistics_core,
                "query": "isBot:true FACET PIVOT statistics_type,type"
            })

    else:
        bot_events_by_type_rows.append({
            "statistics_type": statistics_type,
            "object_type": None,
            "object_type_label": None,
            "bot_status": "bot",
            "events_count": statistics_type_entry.get("count", 0),
            "core_name": statistics_core,
            "query": "isBot:true FACET PIVOT statistics_type,type"
        })

bot_events_by_type_columns = [
    "statistics_type",
    "object_type",
    "object_type_label",
    "bot_status",
    "events_count",
    "core_name",
    "query",
]

df_bot_events_by_type = pd.DataFrame(
    bot_events_by_type_rows,
    columns=bot_events_by_type_columns
)

df_bot_events_by_type["reference_month"] = REPORT_MONTH
df_bot_events_by_type["environment"] = ENVIRONMENT
df_bot_events_by_type["source"] = "solr"
df_bot_events_by_type["metric_definition"] = (
    "bot events in the Solr statistics core grouped by statistics_type "
    "and DSpace object type"
)

output_bot_events_by_type = export_path("solr_bot_events_by_type.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_bot_events_by_type.to_csv(output_bot_events_by_type, index=False)

print(f"Bot event type rows extracted: {len(df_bot_events_by_type)}")
print(f"Total bot events: {df_bot_events_by_type['events_count'].sum()}")

display(
    df_bot_events_by_type
    .sort_values("events_count", ascending=False)
)

print(f"File saved to: {output_bot_events_by_type.resolve()}")

### Bot events in the reference month

In [ ]:
# =========================================================
# BOT EVENTS IN THE REFERENCE MONTH BY EVENT AND OBJECT TYPE
# =========================================================

# - statistics_type
# - object_type
# - object_type_label
# - bot_status
# - events_count
# - core_name
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

statistics_object_type_legend = {
    "0": "bitstream",
    "1": "bundle",
    "2": "item",
    "3": "collection",
    "4": "community",
    "5": "site",
    "6": "group",
    "7": "eperson"
}

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

response_bot_events_reference_month_by_type = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.pivot": "statistics_type,type",
        "facet.mincount": 1,
        "facet.limit": -1,
        "wt": "json"
    }
)

pivot_values = (
    response_bot_events_reference_month_by_type
    .get("facet_counts", {})
    .get("facet_pivot", {})
    .get("statistics_type,type", [])
)

bot_events_reference_month_by_type_rows = []

for statistics_type_entry in pivot_values:
    statistics_type = statistics_type_entry.get("value")
    object_type_pivot = statistics_type_entry.get("pivot", [])

    if object_type_pivot:
        for object_type_entry in object_type_pivot:
            object_type = str(object_type_entry.get("value"))
            events_count = object_type_entry.get("count", 0)

            bot_events_reference_month_by_type_rows.append({
                "statistics_type": statistics_type,
                "object_type": object_type,
                "object_type_label": statistics_object_type_legend.get(
                    object_type,
                    "unknown"
                ),
                "bot_status": "bot",
                "events_count": events_count,
                "core_name": statistics_core,
                "period_start": reference_month_start.strftime("%Y-%m-%d"),
                "period_end": reference_month_end.strftime("%Y-%m-%d"),
                "query": (
                    "isBot:true "
                    f"AND {time_filter} "
                    "FACET PIVOT statistics_type,type"
                )
            })

    else:
        bot_events_reference_month_by_type_rows.append({
            "statistics_type": statistics_type,
            "object_type": None,
            "object_type_label": None,
            "bot_status": "bot",
            "events_count": statistics_type_entry.get("count", 0),
            "core_name": statistics_core,
            "period_start": reference_month_start.strftime("%Y-%m-%d"),
            "period_end": reference_month_end.strftime("%Y-%m-%d"),
            "query": (
                "isBot:true "
                f"AND {time_filter} "
                "FACET PIVOT statistics_type,type"
            )
        })

bot_events_reference_month_by_type_columns = [
    "statistics_type",
    "object_type",
    "object_type_label",
    "bot_status",
    "events_count",
    "core_name",
    "period_start",
    "period_end",
    "query",
]

df_bot_events_reference_month_by_type = pd.DataFrame(
    bot_events_reference_month_by_type_rows,
    columns=bot_events_reference_month_by_type_columns
)

df_bot_events_reference_month_by_type["reference_month"] = REPORT_MONTH
df_bot_events_reference_month_by_type["environment"] = ENVIRONMENT
df_bot_events_reference_month_by_type["source"] = "solr"
df_bot_events_reference_month_by_type["metric_definition"] = (
    "bot events in the Solr statistics core during the reference month, "
    "grouped by statistics_type and DSpace object type"
)

output_bot_events_reference_month_by_type = (
    export_path("solr_bot_events_reference_month_by_type.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_bot_events_reference_month_by_type.to_csv(
    output_bot_events_reference_month_by_type,
    index=False
)

print(
    "Bot event type rows in the reference month extracted: "
    f"{len(df_bot_events_reference_month_by_type)}"
)

if not df_bot_events_reference_month_by_type.empty:
    print(
        "Total bot events in the reference month: "
        f"{df_bot_events_reference_month_by_type['events_count'].sum()}"
    )
else:
    print("Total bot events in the reference month: 0")

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(
    df_bot_events_reference_month_by_type
    .sort_values("events_count", ascending=False)
)

print(f"File saved to: {output_bot_events_reference_month_by_type.resolve()}")

### Most frequent bot user agents

In [ ]:
# =========================================================
# MOST FREQUENT BOT USER AGENTS
# =========================================================

# - user_agent
# - events_count
# - core_name
# - bot_status
# - facet_field
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_most_frequent_bot_user_agents = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "userAgent",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": 50,
        "wt": "json"
    }
)

facet_values = (
    response_most_frequent_bot_user_agents
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("userAgent", [])
)

most_frequent_bot_user_agents_rows = []

for i in range(0, len(facet_values), 2):
    most_frequent_bot_user_agents_rows.append({
        "user_agent": facet_values[i],
        "events_count": facet_values[i + 1],
        "core_name": statistics_core,
        "bot_status": "bot",
        "facet_field": "userAgent",
        "query": "isBot:true FACET userAgent"
    })

most_frequent_bot_user_agents_columns = [
    "user_agent",
    "events_count",
    "core_name",
    "bot_status",
    "facet_field",
    "query",
]

df_most_frequent_bot_user_agents = pd.DataFrame(
    most_frequent_bot_user_agents_rows,
    columns=most_frequent_bot_user_agents_columns
)

df_most_frequent_bot_user_agents["reference_month"] = REPORT_MONTH
df_most_frequent_bot_user_agents["environment"] = ENVIRONMENT
df_most_frequent_bot_user_agents["source"] = "solr"
df_most_frequent_bot_user_agents["metric_definition"] = (
    "most frequent user agents among events marked as bot events "
    "in the Solr statistics core"
)

output_most_frequent_bot_user_agents = (
    export_path("solr_most_frequent_bot_user_agents.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_most_frequent_bot_user_agents.to_csv(
    output_most_frequent_bot_user_agents,
    index=False
)

print(
    "Most frequent bot user agents extracted: "
    f"{len(df_most_frequent_bot_user_agents)}"
)

display(df_most_frequent_bot_user_agents.head())

print(f"File saved to: {output_most_frequent_bot_user_agents.resolve()}")

### Most frequent bot user agents in the reference month

In [ ]:
# =========================================================
# MOST FREQUENT BOT USER AGENTS IN THE REFERENCE MONTH
# =========================================================

# - user_agent
# - events_count
# - core_name
# - bot_status
# - facet_field
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

response_most_frequent_bot_user_agents_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "userAgent",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": 50,
        "wt": "json"
    }
)

facet_values = (
    response_most_frequent_bot_user_agents_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("userAgent", [])
)

most_frequent_bot_user_agents_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    most_frequent_bot_user_agents_reference_month_rows.append({
        "user_agent": facet_values[i],
        "events_count": facet_values[i + 1],
        "core_name": statistics_core,
        "bot_status": "bot",
        "facet_field": "userAgent",
        "period_start": reference_month_start.strftime("%Y-%m-%d"),
        "period_end": reference_month_end.strftime("%Y-%m-%d"),
        "query": (
            "isBot:true "
            f"AND {time_filter} "
            "FACET userAgent"
        )
    })

most_frequent_bot_user_agents_reference_month_columns = [
    "user_agent",
    "events_count",
    "core_name",
    "bot_status",
    "facet_field",
    "period_start",
    "period_end",
    "query",
]

df_most_frequent_bot_user_agents_reference_month = pd.DataFrame(
    most_frequent_bot_user_agents_reference_month_rows,
    columns=most_frequent_bot_user_agents_reference_month_columns
)

df_most_frequent_bot_user_agents_reference_month["reference_month"] = REPORT_MONTH
df_most_frequent_bot_user_agents_reference_month["environment"] = ENVIRONMENT
df_most_frequent_bot_user_agents_reference_month["source"] = "solr"
df_most_frequent_bot_user_agents_reference_month["metric_definition"] = (
    "most frequent user agents among events marked as bot events "
    "in the Solr statistics core during the reference month"
)

output_most_frequent_bot_user_agents_reference_month = (
    export_path("solr_most_frequent_bot_user_agents_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_most_frequent_bot_user_agents_reference_month.to_csv(
    output_most_frequent_bot_user_agents_reference_month,
    index=False
)

print(
    "Most frequent bot user agents in the reference month extracted: "
    f"{len(df_most_frequent_bot_user_agents_reference_month)}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_most_frequent_bot_user_agents_reference_month.head())

print(f"File saved to: {output_most_frequent_bot_user_agents_reference_month.resolve()}")

### Internal events

In [ ]:
# =========================================================
# INTERNAL EVENTS FROM 192.168.X.X NETWORK BY EVENT AND OBJECT TYPE
# =========================================================

# - network_scope
# - ip_pattern
# - statistics_type
# - object_type
# - object_type_label
# - events_count
# - core_name
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

statistics_object_type_legend = {
    "0": "bitstream",
    "1": "bundle",
    "2": "item",
    "3": "collection",
    "4": "community",
    "5": "site",
    "6": "group",
    "7": "eperson"
}

response_internal_events_192_168_by_type = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "ip:192.168.*"
        ],
        "rows": 0,
        "facet": "true",
        "facet.pivot": "statistics_type,type",
        "facet.mincount": 1,
        "facet.limit": -1,
        "wt": "json"
    }
)

pivot_values = (
    response_internal_events_192_168_by_type
    .get("facet_counts", {})
    .get("facet_pivot", {})
    .get("statistics_type,type", [])
)

internal_events_192_168_by_type_rows = []

for statistics_type_entry in pivot_values:
    statistics_type = statistics_type_entry.get("value")
    statistics_type_count = statistics_type_entry.get("count", 0)
    object_type_pivot = statistics_type_entry.get("pivot", [])

    if object_type_pivot:
        for object_type_entry in object_type_pivot:
            object_type = str(object_type_entry.get("value"))
            events_count = object_type_entry.get("count", 0)

            internal_events_192_168_by_type_rows.append({
                "network_scope": "internal",
                "ip_pattern": "192.168.X.X",
                "statistics_type": statistics_type,
                "object_type": object_type,
                "object_type_label": statistics_object_type_legend.get(
                    object_type,
                    "unknown"
                ),
                "events_count": events_count,
                "core_name": statistics_core,
                "query": "ip:192.168.* FACET PIVOT statistics_type,type"
            })

    else:
        internal_events_192_168_by_type_rows.append({
            "network_scope": "internal",
            "ip_pattern": "192.168.X.X",
            "statistics_type": statistics_type,
            "object_type": "not_applicable",
            "object_type_label": "not_applicable",
            "events_count": statistics_type_count,
            "core_name": statistics_core,
            "query": "ip:192.168.* FACET PIVOT statistics_type,type"
        })

internal_events_192_168_by_type_columns = [
    "network_scope",
    "ip_pattern",
    "statistics_type",
    "object_type",
    "object_type_label",
    "events_count",
    "core_name",
    "query",
]

df_internal_events_192_168_by_type = pd.DataFrame(
    internal_events_192_168_by_type_rows,
    columns=internal_events_192_168_by_type_columns
)

df_internal_events_192_168_by_type["reference_month"] = REPORT_MONTH
df_internal_events_192_168_by_type["environment"] = ENVIRONMENT
df_internal_events_192_168_by_type["source"] = "solr"
df_internal_events_192_168_by_type["metric_definition"] = (
    "events in the Solr statistics core whose IP address belongs to the "
    "192.168.X.X private/internal network range, grouped by statistics_type "
    "and DSpace object type"
)

output_internal_events_192_168_by_type = (
    export_path("solr_internal_events_192_168_by_type.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_internal_events_192_168_by_type.to_csv(
    output_internal_events_192_168_by_type,
    index=False
)

print(
    "Internal event type rows from 192.168.X.X network extracted: "
    f"{len(df_internal_events_192_168_by_type)}"
)

if not df_internal_events_192_168_by_type.empty:
    print(
        "Total internal events from 192.168.X.X network: "
        f"{df_internal_events_192_168_by_type['events_count'].sum()}"
    )
else:
    print("Total internal events from 192.168.X.X network: 0")

display(
    df_internal_events_192_168_by_type
    .sort_values("events_count", ascending=False)
)

print(f"File saved to: {output_internal_events_192_168_by_type.resolve()}")

### Internal events in the reference month

In [ ]:
# =========================================================
# INTERNAL EVENTS FROM 192.168.X.X NETWORK IN THE REFERENCE MONTH BY EVENT AND OBJECT TYPE
# =========================================================

# - network_scope
# - ip_pattern
# - statistics_type
# - object_type
# - object_type_label
# - events_count
# - core_name
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

statistics_object_type_legend = {
    "0": "bitstream",
    "1": "bundle",
    "2": "item",
    "3": "collection",
    "4": "community",
    "5": "site",
    "6": "group",
    "7": "eperson"
}

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

response_internal_events_192_168_reference_month_by_type = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "ip:192.168.*",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.pivot": "statistics_type,type",
        "facet.mincount": 1,
        "facet.limit": -1,
        "wt": "json"
    }
)

pivot_values = (
    response_internal_events_192_168_reference_month_by_type
    .get("facet_counts", {})
    .get("facet_pivot", {})
    .get("statistics_type,type", [])
)

internal_events_192_168_reference_month_by_type_rows = []

for statistics_type_entry in pivot_values:
    statistics_type = statistics_type_entry.get("value")
    statistics_type_count = statistics_type_entry.get("count", 0)
    object_type_pivot = statistics_type_entry.get("pivot", [])

    if object_type_pivot:
        for object_type_entry in object_type_pivot:
            object_type = str(object_type_entry.get("value"))
            events_count = object_type_entry.get("count", 0)

            internal_events_192_168_reference_month_by_type_rows.append({
                "network_scope": "internal",
                "ip_pattern": "192.168.X.X",
                "statistics_type": statistics_type,
                "object_type": object_type,
                "object_type_label": statistics_object_type_legend.get(
                    object_type,
                    "unknown"
                ),
                "events_count": events_count,
                "core_name": statistics_core,
                "period_start": reference_month_start.strftime("%Y-%m-%d"),
                "period_end": reference_month_end.strftime("%Y-%m-%d"),
                "query": (
                    "ip:192.168.* "
                    f"AND {time_filter} "
                    "FACET PIVOT statistics_type,type"
                )
            })

    else:
        internal_events_192_168_reference_month_by_type_rows.append({
            "network_scope": "internal",
            "ip_pattern": "192.168.X.X",
            "statistics_type": statistics_type,
            "object_type": "not_applicable",
            "object_type_label": "not_applicable",
            "events_count": statistics_type_count,
            "core_name": statistics_core,
            "period_start": reference_month_start.strftime("%Y-%m-%d"),
            "period_end": reference_month_end.strftime("%Y-%m-%d"),
            "query": (
                "ip:192.168.* "
                f"AND {time_filter} "
                "FACET PIVOT statistics_type,type"
            )
        })

internal_events_192_168_reference_month_by_type_columns = [
    "network_scope",
    "ip_pattern",
    "statistics_type",
    "object_type",
    "object_type_label",
    "events_count",
    "core_name",
    "period_start",
    "period_end",
    "query",
]

df_internal_events_192_168_reference_month_by_type = pd.DataFrame(
    internal_events_192_168_reference_month_by_type_rows,
    columns=internal_events_192_168_reference_month_by_type_columns
)

df_internal_events_192_168_reference_month_by_type["reference_month"] = REPORT_MONTH
df_internal_events_192_168_reference_month_by_type["environment"] = ENVIRONMENT
df_internal_events_192_168_reference_month_by_type["source"] = "solr"
df_internal_events_192_168_reference_month_by_type["metric_definition"] = (
    "events in the Solr statistics core during the reference month whose IP address "
    "belongs to the 192.168.X.X private/internal network range, grouped by "
    "statistics_type and DSpace object type"
)

output_internal_events_192_168_reference_month_by_type = (
    export_path("solr_internal_events_192_168_reference_month_by_type.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_internal_events_192_168_reference_month_by_type.to_csv(
    output_internal_events_192_168_reference_month_by_type,
    index=False
)

print(
    "Internal event type rows from 192.168.X.X network in the reference month extracted: "
    f"{len(df_internal_events_192_168_reference_month_by_type)}"
)

if not df_internal_events_192_168_reference_month_by_type.empty:
    print(
        "Total internal events from 192.168.X.X network in the reference month: "
        f"{df_internal_events_192_168_reference_month_by_type['events_count'].sum()}"
    )
else:
    print("Total internal events from 192.168.X.X network in the reference month: 0")

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(
    df_internal_events_192_168_reference_month_by_type
    .sort_values("events_count", ascending=False)
)

print(f"File saved to: {output_internal_events_192_168_reference_month_by_type.resolve()}")

### Internal / External events ratio

In [ ]:
# =========================================================
# INTERNAL / EXTERNAL EVENTS RATIO BY BOT STATUS AND EVENT TYPE
# =========================================================

# - network_scope
# - bot_status
# - statistics_type
# - object_type
# - object_type_label
# - events_count
# - events_percentage
# - core_name
# - ip_rule
# - bot_rule
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

statistics_object_type_legend = {
    "0": "bitstream",
    "1": "bundle",
    "2": "item",
    "3": "collection",
    "4": "community",
    "5": "site",
    "6": "group",
    "7": "eperson"
}

network_scope_queries = {
    "internal": "ip:192.168.*",
    "external": "ip:[* TO *] AND -ip:192.168.*",
    "unknown": "(*:* NOT ip:[* TO *])"
}

bot_status_queries = {
    "bot": "isBot:true",
    "non_bot": "-isBot:true"
}

network_scope_bot_type_rows = []

for network_scope, scope_query in network_scope_queries.items():
    for bot_status, bot_query in bot_status_queries.items():

        response_network_scope_bot_type = solr_select(
            statistics_core,
            {
                "q": "*:*",
                "fq": [
                    scope_query,
                    bot_query
                ],
                "rows": 0,
                "facet": "true",
                "facet.pivot": "statistics_type,type",
                "facet.mincount": 1,
                "facet.limit": -1,
                "wt": "json"
            }
        )

        pivot_values = (
            response_network_scope_bot_type
            .get("facet_counts", {})
            .get("facet_pivot", {})
            .get("statistics_type,type", [])
        )

        for statistics_type_entry in pivot_values:
            statistics_type = statistics_type_entry.get("value")
            statistics_type_count = statistics_type_entry.get("count", 0)
            object_type_pivot = statistics_type_entry.get("pivot", [])

            if object_type_pivot:
                for object_type_entry in object_type_pivot:
                    object_type = str(object_type_entry.get("value"))
                    events_count = object_type_entry.get("count", 0)

                    network_scope_bot_type_rows.append({
                        "network_scope": network_scope,
                        "bot_status": bot_status,
                        "statistics_type": statistics_type,
                        "object_type": object_type,
                        "object_type_label": statistics_object_type_legend.get(
                            object_type,
                            "unknown"
                        ),
                        "events_count": events_count,
                        "core_name": statistics_core,
                        "ip_rule": scope_query,
                        "bot_rule": bot_query,
                        "query": (
                            f"{scope_query} AND {bot_query} "
                            "FACET PIVOT statistics_type,type"
                        )
                    })

            else:
                network_scope_bot_type_rows.append({
                    "network_scope": network_scope,
                    "bot_status": bot_status,
                    "statistics_type": statistics_type,
                    "object_type": "not_applicable",
                    "object_type_label": "not_applicable",
                    "events_count": statistics_type_count,
                    "core_name": statistics_core,
                    "ip_rule": scope_query,
                    "bot_rule": bot_query,
                    "query": (
                        f"{scope_query} AND {bot_query} "
                        "FACET PIVOT statistics_type,type"
                    )
                })

internal_external_events_ratio_by_bot_status_and_type_columns = [
    "network_scope",
    "bot_status",
    "statistics_type",
    "object_type",
    "object_type_label",
    "events_count",
    "core_name",
    "ip_rule",
    "bot_rule",
    "query",
]

df_internal_external_events_ratio_by_bot_status_and_type = pd.DataFrame(
    network_scope_bot_type_rows,
    columns=internal_external_events_ratio_by_bot_status_and_type_columns
)

total_events_with_scope = (
    df_internal_external_events_ratio_by_bot_status_and_type["events_count"].sum()
)

if total_events_with_scope > 0:
    df_internal_external_events_ratio_by_bot_status_and_type["events_percentage"] = (
        df_internal_external_events_ratio_by_bot_status_and_type["events_count"]
        / total_events_with_scope
        * 100
    ).round(2)
else:
    df_internal_external_events_ratio_by_bot_status_and_type["events_percentage"] = 0

df_internal_external_events_ratio_by_bot_status_and_type["reference_month"] = REPORT_MONTH
df_internal_external_events_ratio_by_bot_status_and_type["environment"] = ENVIRONMENT
df_internal_external_events_ratio_by_bot_status_and_type["source"] = "solr"
df_internal_external_events_ratio_by_bot_status_and_type["metric_definition"] = (
    "ratio between internal, external and unknown events in the Solr statistics core, "
    "split by bot status, statistics_type and DSpace object type; internal events are "
    "identified by IP addresses matching 192.168.X.X"
)

output_internal_external_events_ratio_by_bot_status_and_type = (
    export_path("solr_internal_external_events_ratio_by_bot_status_and_type.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_internal_external_events_ratio_by_bot_status_and_type.to_csv(
    output_internal_external_events_ratio_by_bot_status_and_type,
    index=False
)

print("Internal / external events ratio by bot status and event type:")

display(
    df_internal_external_events_ratio_by_bot_status_and_type
    .sort_values("events_count", ascending=False)
)

print("Distribution by network scope:")
display(
    df_internal_external_events_ratio_by_bot_status_and_type
    .groupby("network_scope", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print("Distribution by bot status:")
display(
    df_internal_external_events_ratio_by_bot_status_and_type
    .groupby("bot_status", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print("Distribution by event type:")
display(
    df_internal_external_events_ratio_by_bot_status_and_type
    .groupby(["statistics_type", "object_type_label"], dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print(f"Total events considered: {total_events_with_scope}")
print(f"File saved to: {output_internal_external_events_ratio_by_bot_status_and_type.resolve()}")

### Internal / External events ratio in the reference month

In [ ]:
# =========================================================
# INTERNAL / EXTERNAL EVENTS RATIO IN THE REFERENCE MONTH BY BOT STATUS AND EVENT TYPE
# =========================================================

# - network_scope
# - bot_status
# - statistics_type
# - object_type
# - object_type_label
# - events_count
# - events_percentage
# - core_name
# - ip_rule
# - bot_rule
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

statistics_object_type_legend = {
    "0": "bitstream",
    "1": "bundle",
    "2": "item",
    "3": "collection",
    "4": "community",
    "5": "site",
    "6": "group",
    "7": "eperson"
}

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

network_scope_queries = {
    "internal": "ip:192.168.*",
    "external": "ip:[* TO *] AND -ip:192.168.*",
    "unknown": "(*:* NOT ip:[* TO *])"
}

bot_status_queries = {
    "bot": "isBot:true",
    "non_bot": "-isBot:true"
}

network_scope_bot_type_reference_month_rows = []

for network_scope, scope_query in network_scope_queries.items():
    for bot_status, bot_query in bot_status_queries.items():

        response_network_scope_bot_type_reference_month = solr_select(
            statistics_core,
            {
                "q": "*:*",
                "fq": [
                    scope_query,
                    bot_query,
                    time_filter
                ],
                "rows": 0,
                "facet": "true",
                "facet.pivot": "statistics_type,type",
                "facet.mincount": 1,
                "facet.limit": -1,
                "wt": "json"
            }
        )

        pivot_values = (
            response_network_scope_bot_type_reference_month
            .get("facet_counts", {})
            .get("facet_pivot", {})
            .get("statistics_type,type", [])
        )

        for statistics_type_entry in pivot_values:
            statistics_type = statistics_type_entry.get("value")
            statistics_type_count = statistics_type_entry.get("count", 0)
            object_type_pivot = statistics_type_entry.get("pivot", [])

            if object_type_pivot:
                for object_type_entry in object_type_pivot:
                    object_type = str(object_type_entry.get("value"))
                    events_count = object_type_entry.get("count", 0)

                    network_scope_bot_type_reference_month_rows.append({
                        "network_scope": network_scope,
                        "bot_status": bot_status,
                        "statistics_type": statistics_type,
                        "object_type": object_type,
                        "object_type_label": statistics_object_type_legend.get(
                            object_type,
                            "unknown"
                        ),
                        "events_count": events_count,
                        "core_name": statistics_core,
                        "ip_rule": scope_query,
                        "bot_rule": bot_query,
                        "period_start": reference_month_start.strftime("%Y-%m-%d"),
                        "period_end": reference_month_end.strftime("%Y-%m-%d"),
                        "query": (
                            f"{scope_query} AND {bot_query} AND {time_filter} "
                            "FACET PIVOT statistics_type,type"
                        )
                    })

            else:
                network_scope_bot_type_reference_month_rows.append({
                    "network_scope": network_scope,
                    "bot_status": bot_status,
                    "statistics_type": statistics_type,
                    "object_type": "not_applicable",
                    "object_type_label": "not_applicable",
                    "events_count": statistics_type_count,
                    "core_name": statistics_core,
                    "ip_rule": scope_query,
                    "bot_rule": bot_query,
                    "period_start": reference_month_start.strftime("%Y-%m-%d"),
                    "period_end": reference_month_end.strftime("%Y-%m-%d"),
                    "query": (
                        f"{scope_query} AND {bot_query} AND {time_filter} "
                        "FACET PIVOT statistics_type,type"
                    )
                })

internal_external_events_ratio_reference_month_by_bot_status_and_type_columns = [
    "network_scope",
    "bot_status",
    "statistics_type",
    "object_type",
    "object_type_label",
    "events_count",
    "core_name",
    "ip_rule",
    "bot_rule",
    "period_start",
    "period_end",
    "query",
]

df_internal_external_events_ratio_reference_month_by_bot_status_and_type = pd.DataFrame(
    network_scope_bot_type_reference_month_rows,
    columns=internal_external_events_ratio_reference_month_by_bot_status_and_type_columns
)

total_events_with_scope_reference_month = (
    df_internal_external_events_ratio_reference_month_by_bot_status_and_type["events_count"].sum()
)

if total_events_with_scope_reference_month > 0:
    df_internal_external_events_ratio_reference_month_by_bot_status_and_type["events_percentage"] = (
        df_internal_external_events_ratio_reference_month_by_bot_status_and_type["events_count"]
        / total_events_with_scope_reference_month
        * 100
    ).round(2)
else:
    df_internal_external_events_ratio_reference_month_by_bot_status_and_type["events_percentage"] = 0

df_internal_external_events_ratio_reference_month_by_bot_status_and_type["reference_month"] = REPORT_MONTH
df_internal_external_events_ratio_reference_month_by_bot_status_and_type["environment"] = ENVIRONMENT
df_internal_external_events_ratio_reference_month_by_bot_status_and_type["source"] = "solr"
df_internal_external_events_ratio_reference_month_by_bot_status_and_type["metric_definition"] = (
    "ratio between internal, external and unknown events in the Solr statistics core "
    "during the reference month, split by bot status, statistics_type and DSpace object type; "
    "internal events are identified by IP addresses matching 192.168.X.X"
)

output_internal_external_events_ratio_reference_month_by_bot_status_and_type = (
    export_path("solr_internal_external_events_ratio_reference_month_by_bot_status_and_type.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_internal_external_events_ratio_reference_month_by_bot_status_and_type.to_csv(
    output_internal_external_events_ratio_reference_month_by_bot_status_and_type,
    index=False
)

print("Internal / external events ratio in the reference month by bot status and event type:")

display(
    df_internal_external_events_ratio_reference_month_by_bot_status_and_type
    .sort_values("events_count", ascending=False)
)

print("Distribution by network scope:")
display(
    df_internal_external_events_ratio_reference_month_by_bot_status_and_type
    .groupby("network_scope", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print("Distribution by bot status:")
display(
    df_internal_external_events_ratio_reference_month_by_bot_status_and_type
    .groupby("bot_status", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print("Distribution by event type:")
display(
    df_internal_external_events_ratio_reference_month_by_bot_status_and_type
    .groupby(["statistics_type", "object_type_label"], dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print(f"Total events considered: {total_events_with_scope_reference_month}")
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

print(
    "File saved to: "
    f"{output_internal_external_events_ratio_reference_month_by_bot_status_and_type.resolve()}"
)

## **Search Events**

### Total searches

In [ ]:
# =========================================================
# TOTAL SEARCHES BY BOT STATUS
# =========================================================

# - metric_name
# - core_name
# - statistics_type
# - bot_status
# - total_events
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

search_bot_status_queries = {
    "bot": "isBot:true",
    "non_bot": "-isBot:true"
}

total_searches_by_bot_status_rows = []

for bot_status, bot_query in search_bot_status_queries.items():
    response_total_searches_by_bot_status = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "statistics_type:search",
                bot_query
            ],
            "rows": 0,
            "wt": "json"
        }
    )

    total_events = (
        response_total_searches_by_bot_status
        .get("response", {})
        .get("numFound", 0)
    )

    total_searches_by_bot_status_rows.append({
        "metric_name": "total_searches",
        "core_name": statistics_core,
        "statistics_type": "search",
        "bot_status": bot_status,
        "total_events": total_events,
        "bot_filter": bot_status,
        "query": f"statistics_type:search AND {bot_query}"
    })

df_total_searches_by_bot_status = pd.DataFrame(
    total_searches_by_bot_status_rows
)

df_total_searches_by_bot_status["reference_month"] = REPORT_MONTH
df_total_searches_by_bot_status["environment"] = ENVIRONMENT
df_total_searches_by_bot_status["source"] = "solr"
df_total_searches_by_bot_status["metric_definition"] = (
    "total number of search events recorded in the Solr statistics core, "
    "split by bot status using the isBot field"
)

output_total_searches_by_bot_status = (
    export_path("solr_total_searches_by_bot_status.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_total_searches_by_bot_status.to_csv(
    output_total_searches_by_bot_status,
    index=False
)

print("Total searches by bot status:")
display(
    df_total_searches_by_bot_status
    .sort_values("total_events", ascending=False)
)

print(
    "Total searches: "
    f"{df_total_searches_by_bot_status['total_events'].sum()}"
)

print(f"File saved to: {output_total_searches_by_bot_status.resolve()}")

### Total searches in the reference month

In [ ]:
# =========================================================
# TOTAL SEARCHES IN THE REFERENCE MONTH BY BOT STATUS
# =========================================================

# - metric_name
# - core_name
# - statistics_type
# - bot_status
# - total_events
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

search_bot_status_queries = {
    "bot": "isBot:true",
    "non_bot": "-isBot:true"
}

total_searches_reference_month_rows = []

for bot_status, bot_query in search_bot_status_queries.items():
    response_total_searches_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "statistics_type:search",
                bot_query,
                time_filter
            ],
            "rows": 0,
            "wt": "json"
        }
    )

    total_events = (
        response_total_searches_reference_month
        .get("response", {})
        .get("numFound", 0)
    )

    total_searches_reference_month_rows.append({
        "metric_name": "total_searches_reference_month",
        "core_name": statistics_core,
        "statistics_type": "search",
        "bot_status": bot_status,
        "total_events": total_events,
        "bot_filter": bot_status,
        "period_start": reference_month_start.strftime("%Y-%m-%d"),
        "period_end": reference_month_end.strftime("%Y-%m-%d"),
        "query": f"statistics_type:search AND {bot_query} AND {time_filter}"
    })

df_total_searches_reference_month = pd.DataFrame(
    total_searches_reference_month_rows
)

df_total_searches_reference_month["reference_month"] = REPORT_MONTH
df_total_searches_reference_month["environment"] = ENVIRONMENT
df_total_searches_reference_month["source"] = "solr"
df_total_searches_reference_month["metric_definition"] = (
    "total number of search events recorded in the Solr statistics core during "
    "the reference month, split by bot status using the isBot field"
)

output_total_searches_reference_month = (
    export_path("solr_total_searches_reference_month_by_bot_status.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_total_searches_reference_month.to_csv(
    output_total_searches_reference_month,
    index=False
)

print("Total searches in the reference month by bot status:")
display(
    df_total_searches_reference_month
    .sort_values("total_events", ascending=False)
)

print(
    "Total searches in the reference month: "
    f"{df_total_searches_reference_month['total_events'].sum()}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

print(f"File saved to: {output_total_searches_reference_month.resolve()}")

### Total searches with details

In [ ]:
# =========================================================
# TOTAL SEARCHES - EVENT DETAILS
# =========================================================

# - search_time
# - search_date
# - search_query
# - search_query_text
# - search_filters
# - search_page
# - search_description
# - ip
# - dns
# - user_agent
# - referrer
# - is_bot
# - bot_status
# - core_name
# - statistics_type
# - query
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse, parse_qsl

statistics_core = SOLR_CORES.get("statistics", "statistics")


def parse_dspace_search_referrer(referrer):
    """
    Parses a DSpace Angular search URL and returns a readable search description.

    If no textual query is available, search_query_text is populated with the
    available search information, such as filters, page number, or a readable
    fallback string.
    """

    if referrer is None or str(referrer).strip() == "":
        fallback_value = "direct_or_unknown"

        return {
            "search_query": fallback_value,
            "search_query_text": fallback_value,
            "search_filters": None,
            "search_page": None,
            "search_description": (
                "Search event without referrer; the original search URL is not available."
            )
        }

    referrer_string = str(referrer).strip()
    parsed_url = urlparse(referrer_string)
    query_params = parse_qsl(parsed_url.query, keep_blank_values=True)

    query_text = None
    page = None
    filters = []
    other_params = []

    for key, value in query_params:
        value = str(value).strip() if value is not None else ""

        if key == "query":
            if value != "":
                query_text = value

        elif key == "spc.page":
            if value != "":
                page = value

        elif key.startswith("f."):
            filter_name = key.replace("f.", "", 1)
            filter_value = value

            if filter_value.endswith(",equals"):
                filter_value = filter_value[:-len(",equals")]

            if filter_value != "":
                filters.append(f"{filter_name}={filter_value}")

        else:
            if value != "":
                other_params.append(f"{key}={value}")
            else:
                other_params.append(key)

    search_parts = []

    if query_text:
        search_parts.append(f"query={query_text}")

    if filters:
        search_parts.append("filters: " + "; ".join(filters))

    if page:
        search_parts.append(f"page={page}")

    if other_params:
        search_parts.append("other_params: " + "; ".join(other_params))

    if search_parts:
        search_query = " | ".join(search_parts)
    else:
        search_query = "search_without_query_or_filters"

    # search_query_text must never be NaN/None.
    # If there is no textual query, use the available structured search information.
    if query_text:
        search_query_text = query_text
    elif filters:
        search_query_text = "filters: " + "; ".join(filters)
    elif page:
        search_query_text = f"search_without_query_or_filters; page={page}"
    elif other_params:
        search_query_text = (
            "search_without_query_or_filters; other_params: "
            + "; ".join(other_params)
        )
    else:
        search_query_text = "search_without_query_or_filters"

    if query_text and filters:
        search_description = (
            f"Textual search for '{query_text}' combined with filters: "
            f"{'; '.join(filters)}"
        )

        if page:
            search_description += f"; results page {page}"

    elif query_text:
        search_description = f"Textual search for '{query_text}'"

        if page:
            search_description += f"; results page {page}"

    elif filters:
        search_description = (
            "Faceted search using filters: "
            f"{'; '.join(filters)}"
        )

        if page:
            search_description += f"; results page {page}"

    elif page:
        search_description = (
            "Search page navigation without textual query or filters; "
            f"only pagination parameter detected: page={page}"
        )

    elif other_params:
        search_description = (
            "Search page event without textual query or recognized filters; "
            f"other parameters detected: {'; '.join(other_params)}"
        )

    else:
        search_description = (
            "Search page event without textual query, filters, or pagination parameters."
        )

    return {
        "search_query": search_query,
        "search_query_text": search_query_text,
        "search_filters": "; ".join(filters) if filters else None,
        "search_page": page,
        "search_description": search_description
    }


# First request: get total number of search events.
response_total_searches = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:search"
        ],
        "rows": 0,
        "wt": "json"
    }
)

total_search_events = (
    response_total_searches
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
search_event_rows = []

for start in range(0, total_search_events, rows_per_page):
    response_search_events = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "statistics_type:search"
            ],
            "rows": rows_per_page,
            "start": start,
            "sort": "time asc",
            "fl": "time,ip,dns,userAgent,referrer,isBot,statistics_type",
            "wt": "json"
        }
    )

    docs = (
        response_search_events
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        is_bot = doc.get("isBot")

        if is_bot is True or str(is_bot).lower() == "true":
            bot_status = "bot"
        elif is_bot is False or str(is_bot).lower() == "false":
            bot_status = "non_bot"
        else:
            bot_status = "unknown"

        parsed_search = parse_dspace_search_referrer(doc.get("referrer"))

        search_time = doc.get("time")

        search_event_rows.append({
            "search_time": search_time,
            "search_date": pd.to_datetime(search_time).date() if search_time else None,
            "search_query": parsed_search["search_query"],
            "search_query_text": parsed_search["search_query_text"],
            "search_filters": parsed_search["search_filters"],
            "search_page": parsed_search["search_page"],
            "search_description": parsed_search["search_description"],
            "ip": doc.get("ip"),
            "dns": doc.get("dns"),
            "user_agent": doc.get("userAgent"),
            "referrer": doc.get("referrer"),
            "is_bot": is_bot,
            "bot_status": bot_status,
            "core_name": statistics_core,
            "statistics_type": doc.get("statistics_type"),
            "query": "statistics_type:search SORT time asc"
        })

search_event_columns = [
    "search_time",
    "search_date",
    "search_query",
    "search_query_text",
    "search_filters",
    "search_page",
    "search_description",
    "ip",
    "dns",
    "user_agent",
    "referrer",
    "is_bot",
    "bot_status",
    "core_name",
    "statistics_type",
    "query",
]

df_total_searches_details = pd.DataFrame(
    search_event_rows,
    columns=search_event_columns
)

df_total_searches_details["reference_month"] = REPORT_MONTH
df_total_searches_details["environment"] = ENVIRONMENT
df_total_searches_details["source"] = "solr"
df_total_searches_details["metric_definition"] = (
    "all search events recorded in the Solr statistics core, including event date, "
    "parsed search query information from referrer URLs, and bot status"
)

output_total_searches_details = export_path("solr_total_searches_details.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_total_searches_details.to_csv(output_total_searches_details, index=False)

print(f"Total search events extracted: {len(df_total_searches_details)}")

print("Distribution by bot status:")
display(
    df_total_searches_details
    .groupby("bot_status", dropna=False)
    .size()
    .reset_index(name="events_count")
    .sort_values("events_count", ascending=False)
)

print("Search events by date:")
display(
    df_total_searches_details
    .groupby("search_date", dropna=False)
    .size()
    .reset_index(name="events_count")
    .sort_values("search_date")
)

print("Most frequent parsed search values:")
display(
    df_total_searches_details
    .groupby(["search_query", "search_query_text", "search_description"], dropna=False)
    .size()
    .reset_index(name="events_count")
    .sort_values("events_count", ascending=False)
    .head(30)
)

display(df_total_searches_details.head())

print(f"File saved to: {output_total_searches_details.resolve()}")

### Total searches with details in the reference month

In [ ]:
# =========================================================
# TOTAL SEARCHES - EVENT DETAILS IN THE REFERENCE MONTH
# =========================================================

# - search_time
# - search_date
# - search_query
# - search_query_text
# - search_filters
# - search_page
# - search_description
# - ip
# - dns
# - user_agent
# - referrer
# - is_bot
# - bot_status
# - core_name
# - statistics_type
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse, parse_qsl

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"


def parse_dspace_search_referrer(referrer):
    """
    Parses a DSpace Angular search URL and returns a readable search description.

    If no textual query is available, search_query_text is populated with the
    available search information, such as filters, page number, or a readable
    fallback string.
    """

    if referrer is None or str(referrer).strip() == "":
        fallback_value = "direct_or_unknown"

        return {
            "search_query": fallback_value,
            "search_query_text": fallback_value,
            "search_filters": None,
            "search_page": None,
            "search_description": (
                "Search event without referrer; the original search URL is not available."
            )
        }

    referrer_string = str(referrer).strip()
    parsed_url = urlparse(referrer_string)
    query_params = parse_qsl(parsed_url.query, keep_blank_values=True)

    query_text = None
    page = None
    filters = []
    other_params = []

    for key, value in query_params:
        value = str(value).strip() if value is not None else ""

        if key == "query":
            if value != "":
                query_text = value

        elif key == "spc.page":
            if value != "":
                page = value

        elif key.startswith("f."):
            filter_name = key.replace("f.", "", 1)
            filter_value = value

            if filter_value.endswith(",equals"):
                filter_value = filter_value[:-len(",equals")]

            if filter_value != "":
                filters.append(f"{filter_name}={filter_value}")

        else:
            if value != "":
                other_params.append(f"{key}={value}")
            else:
                other_params.append(key)

    search_parts = []

    if query_text:
        search_parts.append(f"query={query_text}")

    if filters:
        search_parts.append("filters: " + "; ".join(filters))

    if page:
        search_parts.append(f"page={page}")

    if other_params:
        search_parts.append("other_params: " + "; ".join(other_params))

    if search_parts:
        search_query = " | ".join(search_parts)
    else:
        search_query = "search_without_query_or_filters"

    # search_query_text must never be NaN/None.
    # If there is no textual query, use the available structured search information.
    if query_text:
        search_query_text = query_text
    elif filters:
        search_query_text = "filters: " + "; ".join(filters)
    elif page:
        search_query_text = f"search_without_query_or_filters; page={page}"
    elif other_params:
        search_query_text = (
            "search_without_query_or_filters; other_params: "
            + "; ".join(other_params)
        )
    else:
        search_query_text = "search_without_query_or_filters"

    if query_text and filters:
        search_description = (
            f"Textual search for '{query_text}' combined with filters: "
            f"{'; '.join(filters)}"
        )

        if page:
            search_description += f"; results page {page}"

    elif query_text:
        search_description = f"Textual search for '{query_text}'"

        if page:
            search_description += f"; results page {page}"

    elif filters:
        search_description = (
            "Faceted search using filters: "
            f"{'; '.join(filters)}"
        )

        if page:
            search_description += f"; results page {page}"

    elif page:
        search_description = (
            "Search page navigation without textual query or filters; "
            f"only pagination parameter detected: page={page}"
        )

    elif other_params:
        search_description = (
            "Search page event without textual query or recognized filters; "
            f"other parameters detected: {'; '.join(other_params)}"
        )

    else:
        search_description = (
            "Search page event without textual query, filters, or pagination parameters."
        )

    return {
        "search_query": search_query,
        "search_query_text": search_query_text,
        "search_filters": "; ".join(filters) if filters else None,
        "search_page": page,
        "search_description": search_description
    }


# First request: get total number of search events in the reference month.
response_total_searches_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:search",
            time_filter
        ],
        "rows": 0,
        "wt": "json"
    }
)

total_search_events_reference_month = (
    response_total_searches_reference_month
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
search_event_reference_month_rows = []

for start in range(0, total_search_events_reference_month, rows_per_page):
    response_search_events_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "statistics_type:search",
                time_filter
            ],
            "rows": rows_per_page,
            "start": start,
            "sort": "time asc",
            "fl": "time,ip,dns,userAgent,referrer,isBot,statistics_type",
            "wt": "json"
        }
    )

    docs = (
        response_search_events_reference_month
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        is_bot = doc.get("isBot")

        if is_bot is True or str(is_bot).lower() == "true":
            bot_status = "bot"
        elif is_bot is False or str(is_bot).lower() == "false":
            bot_status = "non_bot"
        else:
            bot_status = "unknown"

        parsed_search = parse_dspace_search_referrer(doc.get("referrer"))

        search_time = doc.get("time")

        search_event_reference_month_rows.append({
            "search_time": search_time,
            "search_date": pd.to_datetime(search_time).date() if search_time else None,
            "search_query": parsed_search["search_query"],
            "search_query_text": parsed_search["search_query_text"],
            "search_filters": parsed_search["search_filters"],
            "search_page": parsed_search["search_page"],
            "search_description": parsed_search["search_description"],
            "ip": doc.get("ip"),
            "dns": doc.get("dns"),
            "user_agent": doc.get("userAgent"),
            "referrer": doc.get("referrer"),
            "is_bot": is_bot,
            "bot_status": bot_status,
            "core_name": statistics_core,
            "statistics_type": doc.get("statistics_type"),
            "period_start": reference_month_start.strftime("%Y-%m-%d"),
            "period_end": reference_month_end.strftime("%Y-%m-%d"),
            "query": (
                "statistics_type:search "
                f"AND {time_filter} "
                "SORT time asc"
            )
        })

search_event_reference_month_columns = [
    "search_time",
    "search_date",
    "search_query",
    "search_query_text",
    "search_filters",
    "search_page",
    "search_description",
    "ip",
    "dns",
    "user_agent",
    "referrer",
    "is_bot",
    "bot_status",
    "core_name",
    "statistics_type",
    "period_start",
    "period_end",
    "query",
]

df_total_searches_details_reference_month = pd.DataFrame(
    search_event_reference_month_rows,
    columns=search_event_reference_month_columns
)

df_total_searches_details_reference_month["reference_month"] = REPORT_MONTH
df_total_searches_details_reference_month["environment"] = ENVIRONMENT
df_total_searches_details_reference_month["source"] = "solr"
df_total_searches_details_reference_month["metric_definition"] = (
    "all search events recorded in the Solr statistics core during the reference month, "
    "including event date, parsed search query information from referrer URLs, and bot status"
)

output_total_searches_details_reference_month = (
    export_path("solr_total_searches_details_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_total_searches_details_reference_month.to_csv(
    output_total_searches_details_reference_month,
    index=False
)

print(
    "Total search events in the reference month extracted: "
    f"{len(df_total_searches_details_reference_month)}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

print("Distribution by bot status:")
display(
    df_total_searches_details_reference_month
    .groupby("bot_status", dropna=False)
    .size()
    .reset_index(name="events_count")
    .sort_values("events_count", ascending=False)
)

print("Search events by date:")
display(
    df_total_searches_details_reference_month
    .groupby("search_date", dropna=False)
    .size()
    .reset_index(name="events_count")
    .sort_values("search_date")
)

print("Most frequent parsed search values:")
display(
    df_total_searches_details_reference_month
    .groupby(["search_query", "search_query_text", "search_description"], dropna=False)
    .size()
    .reset_index(name="events_count")
    .sort_values("events_count", ascending=False)
    .head(30)
)

display(df_total_searches_details_reference_month.head())

print(f"File saved to: {output_total_searches_details_reference_month.resolve()}")

### Most searched terms

In [ ]:
# =========================================================
# MOST SEARCHED TERMS
# =========================================================

# - search_query
# - searched_term
# - term_type
# - events_count
# - first_search_time
# - last_search_time
# - first_search_date
# - last_search_date
# - bot_status
# - core_name
# - statistics_type
# - query
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse, parse_qsl

statistics_core = SOLR_CORES.get("statistics", "statistics")


def parse_dspace_search_terms(referrer):
    """
    Parses a DSpace Angular search URL and extracts searchable terms
    together with the full readable search query.
    """

    if referrer is None or str(referrer).strip() == "":
        return {
            "search_query": "direct_or_unknown",
            "terms": [
                {
                    "searched_term": "direct_or_unknown",
                    "term_type": "direct_or_unknown"
                }
            ]
        }

    parsed_url = urlparse(str(referrer).strip())
    query_params = parse_qsl(parsed_url.query, keep_blank_values=True)

    terms = []
    query_text = None
    page = None
    filters = []
    other_params = []
    has_query_or_filter = False

    for key, value in query_params:
        value = str(value).strip() if value is not None else ""

        if key == "query":
            if value != "":
                query_text = value
                has_query_or_filter = True
                terms.append({
                    "searched_term": value,
                    "term_type": "query"
                })

        elif key == "spc.page":
            if value != "":
                page = value

        elif key.startswith("f."):
            filter_name = key.replace("f.", "", 1)
            filter_value = value

            if filter_value.endswith(",equals"):
                filter_value = filter_value[:-len(",equals")]

            if filter_value != "":
                has_query_or_filter = True
                filters.append(f"{filter_name}={filter_value}")
                terms.append({
                    "searched_term": filter_value,
                    "term_type": f"filter:{filter_name}"
                })

        else:
            if value != "":
                other_params.append(f"{key}={value}")
            else:
                other_params.append(key)

    search_parts = []

    if query_text:
        search_parts.append(f"query={query_text}")

    if filters:
        search_parts.append("filters: " + "; ".join(filters))

    if page:
        search_parts.append(f"page={page}")

    if other_params:
        search_parts.append("other_params: " + "; ".join(other_params))

    if search_parts:
        search_query = " | ".join(search_parts)
    else:
        search_query = "search_without_query_or_filters"

    if not has_query_or_filter:
        if page:
            terms.append({
                "searched_term": f"search_without_query_or_filters; page={page}",
                "term_type": "page_only"
            })
        else:
            terms.append({
                "searched_term": "search_without_query_or_filters",
                "term_type": "empty_search"
            })

    return {
        "search_query": search_query,
        "terms": terms
    }


response_total_searches = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:search"
        ],
        "rows": 0,
        "wt": "json"
    }
)

total_search_events = (
    response_total_searches
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
searched_terms_rows = []

for start in range(0, total_search_events, rows_per_page):
    response_search_events = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "statistics_type:search"
            ],
            "rows": rows_per_page,
            "start": start,
            "sort": "time asc",
            "fl": "time,referrer,isBot,statistics_type",
            "wt": "json"
        }
    )

    docs = (
        response_search_events
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        is_bot = doc.get("isBot")

        if is_bot is True or str(is_bot).lower() == "true":
            bot_status = "bot"
        elif is_bot is False or str(is_bot).lower() == "false":
            bot_status = "non_bot"
        else:
            bot_status = "unknown"

        search_time = doc.get("time")
        parsed_search = parse_dspace_search_terms(doc.get("referrer"))

        for term in parsed_search["terms"]:
            searched_terms_rows.append({
                "search_query": parsed_search["search_query"],
                "searched_term": term["searched_term"],
                "term_type": term["term_type"],
                "search_time": search_time,
                "search_date": pd.to_datetime(search_time).date() if search_time else None,
                "bot_status": bot_status,
                "core_name": statistics_core,
                "statistics_type": doc.get("statistics_type"),
                "query": "statistics_type:search SORT time asc"
            })

df_searched_terms_events = pd.DataFrame(searched_terms_rows)

if not df_searched_terms_events.empty:
    df_most_searched_terms = (
        df_searched_terms_events
        .groupby(
            [
                "search_query",
                "searched_term",
                "term_type",
                "bot_status",
                "core_name",
                "statistics_type",
                "query"
            ],
            dropna=False
        )
        .agg(
            events_count=("searched_term", "size"),
            first_search_time=("search_time", "min"),
            last_search_time=("search_time", "max"),
            first_search_date=("search_date", "min"),
            last_search_date=("search_date", "max")
        )
        .reset_index()
        .sort_values("events_count", ascending=False)
    )
else:
    df_most_searched_terms = pd.DataFrame(
        columns=[
            "search_query",
            "searched_term",
            "term_type",
            "bot_status",
            "core_name",
            "statistics_type",
            "query",
            "events_count",
            "first_search_time",
            "last_search_time",
            "first_search_date",
            "last_search_date"
        ]
    )

df_most_searched_terms["reference_month"] = REPORT_MONTH
df_most_searched_terms["environment"] = ENVIRONMENT
df_most_searched_terms["source"] = "solr"
df_most_searched_terms["metric_definition"] = (
    "most searched terms extracted from DSpace search referrer URLs, grouped by complete "
    "search query, term type and bot status; includes the first and last date/time in which "
    "each term was searched"
)

output_most_searched_terms = export_path("solr_most_searched_terms.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_most_searched_terms.to_csv(output_most_searched_terms, index=False)

print(f"Most searched term groups extracted: {len(df_most_searched_terms)}")

print("Distribution by term type:")
display(
    df_most_searched_terms
    .groupby("term_type", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print("Distribution by bot status:")
display(
    df_most_searched_terms
    .groupby("bot_status", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

display(df_most_searched_terms.head(30))

print(f"File saved to: {output_most_searched_terms.resolve()}")

### Most searched terms in the reference month

In [ ]:
# =========================================================
# MOST SEARCHED TERMS IN THE REFERENCE MONTH
# =========================================================

# - search_query
# - searched_term
# - term_type
# - events_count
# - first_search_time
# - last_search_time
# - first_search_date
# - last_search_date
# - bot_status
# - core_name
# - statistics_type
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse, parse_qsl

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"


def parse_dspace_search_terms(referrer):
    """
    Parses a DSpace Angular search URL and extracts searchable terms
    together with the full readable search query.
    """

    if referrer is None or str(referrer).strip() == "":
        return {
            "search_query": "direct_or_unknown",
            "terms": [
                {
                    "searched_term": "direct_or_unknown",
                    "term_type": "direct_or_unknown"
                }
            ]
        }

    parsed_url = urlparse(str(referrer).strip())
    query_params = parse_qsl(parsed_url.query, keep_blank_values=True)

    terms = []
    query_text = None
    page = None
    filters = []
    other_params = []
    has_query_or_filter = False

    for key, value in query_params:
        value = str(value).strip() if value is not None else ""

        if key == "query":
            if value != "":
                query_text = value
                has_query_or_filter = True
                terms.append({
                    "searched_term": value,
                    "term_type": "query"
                })

        elif key == "spc.page":
            if value != "":
                page = value

        elif key.startswith("f."):
            filter_name = key.replace("f.", "", 1)
            filter_value = value

            if filter_value.endswith(",equals"):
                filter_value = filter_value[:-len(",equals")]

            if filter_value != "":
                has_query_or_filter = True
                filters.append(f"{filter_name}={filter_value}")
                terms.append({
                    "searched_term": filter_value,
                    "term_type": f"filter:{filter_name}"
                })

        else:
            if value != "":
                other_params.append(f"{key}={value}")
            else:
                other_params.append(key)

    search_parts = []

    if query_text:
        search_parts.append(f"query={query_text}")

    if filters:
        search_parts.append("filters: " + "; ".join(filters))

    if page:
        search_parts.append(f"page={page}")

    if other_params:
        search_parts.append("other_params: " + "; ".join(other_params))

    if search_parts:
        search_query = " | ".join(search_parts)
    else:
        search_query = "search_without_query_or_filters"

    if not has_query_or_filter:
        if page:
            terms.append({
                "searched_term": f"search_without_query_or_filters; page={page}",
                "term_type": "page_only"
            })
        else:
            terms.append({
                "searched_term": "search_without_query_or_filters",
                "term_type": "empty_search"
            })

    return {
        "search_query": search_query,
        "terms": terms
    }


response_total_searches_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "statistics_type:search",
            time_filter
        ],
        "rows": 0,
        "wt": "json"
    }
)

total_search_events_reference_month = (
    response_total_searches_reference_month
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
searched_terms_reference_month_rows = []

for start in range(0, total_search_events_reference_month, rows_per_page):
    response_search_events_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "statistics_type:search",
                time_filter
            ],
            "rows": rows_per_page,
            "start": start,
            "sort": "time asc",
            "fl": "time,referrer,isBot,statistics_type",
            "wt": "json"
        }
    )

    docs = (
        response_search_events_reference_month
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        is_bot = doc.get("isBot")

        if is_bot is True or str(is_bot).lower() == "true":
            bot_status = "bot"
        elif is_bot is False or str(is_bot).lower() == "false":
            bot_status = "non_bot"
        else:
            bot_status = "unknown"

        search_time = doc.get("time")
        parsed_search = parse_dspace_search_terms(doc.get("referrer"))

        for term in parsed_search["terms"]:
            searched_terms_reference_month_rows.append({
                "search_query": parsed_search["search_query"],
                "searched_term": term["searched_term"],
                "term_type": term["term_type"],
                "search_time": search_time,
                "search_date": pd.to_datetime(search_time).date() if search_time else None,
                "bot_status": bot_status,
                "core_name": statistics_core,
                "statistics_type": doc.get("statistics_type"),
                "period_start": reference_month_start.strftime("%Y-%m-%d"),
                "period_end": reference_month_end.strftime("%Y-%m-%d"),
                "query": (
                    "statistics_type:search "
                    f"AND {time_filter} "
                    "SORT time asc"
                )
            })

df_searched_terms_reference_month_events = pd.DataFrame(
    searched_terms_reference_month_rows
)

if not df_searched_terms_reference_month_events.empty:
    df_most_searched_terms_reference_month = (
        df_searched_terms_reference_month_events
        .groupby(
            [
                "search_query",
                "searched_term",
                "term_type",
                "bot_status",
                "core_name",
                "statistics_type",
                "period_start",
                "period_end",
                "query"
            ],
            dropna=False
        )
        .agg(
            events_count=("searched_term", "size"),
            first_search_time=("search_time", "min"),
            last_search_time=("search_time", "max"),
            first_search_date=("search_date", "min"),
            last_search_date=("search_date", "max")
        )
        .reset_index()
        .sort_values("events_count", ascending=False)
    )
else:
    df_most_searched_terms_reference_month = pd.DataFrame(
        columns=[
            "search_query",
            "searched_term",
            "term_type",
            "bot_status",
            "core_name",
            "statistics_type",
            "period_start",
            "period_end",
            "query",
            "events_count",
            "first_search_time",
            "last_search_time",
            "first_search_date",
            "last_search_date"
        ]
    )

df_most_searched_terms_reference_month["reference_month"] = REPORT_MONTH
df_most_searched_terms_reference_month["environment"] = ENVIRONMENT
df_most_searched_terms_reference_month["source"] = "solr"
df_most_searched_terms_reference_month["metric_definition"] = (
    "most searched terms extracted from DSpace search referrer URLs during the reference month, "
    "grouped by complete search query, term type and bot status; includes the first and last "
    "date/time in which each term was searched in the reference month"
)

output_most_searched_terms_reference_month = (
    export_path("solr_most_searched_terms_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_most_searched_terms_reference_month.to_csv(
    output_most_searched_terms_reference_month,
    index=False
)

print(
    "Most searched term groups in the reference month extracted: "
    f"{len(df_most_searched_terms_reference_month)}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

print("Distribution by term type:")
display(
    df_most_searched_terms_reference_month
    .groupby("term_type", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

print("Distribution by bot status:")
display(
    df_most_searched_terms_reference_month
    .groupby("bot_status", dropna=False)["events_count"]
    .sum()
    .reset_index()
    .sort_values("events_count", ascending=False)
)

display(df_most_searched_terms_reference_month.head(30))

print(f"File saved to: {output_most_searched_terms_reference_month.resolve()}")

## **OAI Core**

### Total OAI records

In [ ]:
# =========================================================
# TOTAL OAI RECORDS
# =========================================================

# - item_id
# - item_handle
# - oai_record_id
# - item_accessioned_date
# - item_available_date
# - resource_issued_date
# - item_last_modified
# - item_deleted
# - item_public
# - item_collections
# - item_communities
# - title
# - authors
# - resource_type
# - language
# - mimetype
# - core_name
# - query
# - reference_month
# - environment
# - source
# - metric_definition

oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_value(value):
    """
    Normalizes Solr scalar/list values for CSV export.
    """

    if isinstance(value, list):
        return "; ".join(str(v) for v in value)

    return value


response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
oai_record_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        oai_record_rows.append({
            "item_id": doc.get("item.id"),
            "item_handle": doc.get("item.handle"),
            "oai_record_id": doc.get("id"),
            "item_accessioned_date": normalize_solr_value(
                doc.get("metadata.dc.date.accessioned")
            ),
            "item_available_date": normalize_solr_value(
                doc.get("metadata.dc.date.available")
            ),
            "resource_issued_date": normalize_solr_value(
                doc.get("metadata.dc.date.issued")
            ),
            "item_last_modified": doc.get("item.lastmodified"),
            "item_deleted": doc.get("item.deleted"),
            "item_public": doc.get("item.public"),
            "item_collections": normalize_solr_value(
                doc.get("item.collections")
            ),
            "item_communities": normalize_solr_value(
                doc.get("item.communities")
            ),
            "title": normalize_solr_value(
                doc.get("metadata.dc.title")
            ),
            "authors": normalize_solr_value(
                doc.get("metadata.dc.contributor.author")
            ),
            "resource_type": normalize_solr_value(
                doc.get("metadata.dc.type")
            ),
            "language": normalize_solr_value(
                doc.get("metadata.dc.language.iso")
            ),
            "mimetype": normalize_solr_value(
                doc.get("metadata.dc.format.mimetype")
            ),
            "core_name": oai_core,
            "query": "*:*"
        })

df_total_oai_records = pd.DataFrame(oai_record_rows)

df_total_oai_records["reference_month"] = REPORT_MONTH
df_total_oai_records["environment"] = ENVIRONMENT
df_total_oai_records["source"] = "solr"
df_total_oai_records["metric_definition"] = (
    "all records indexed in the Solr OAI core, including item_id, handle, "
    "accession/deposit date, availability date, issued date, last modification date, "
    "public/deleted status and selected descriptive metadata"
)

output_total_oai_records = export_path("solr_total_oai_records.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_total_oai_records.to_csv(output_total_oai_records, index=False)

print(f"Total OAI records extracted: {len(df_total_oai_records)}")

print("Distribution by item_public:")
display(
    df_total_oai_records["item_public"]
    .value_counts(dropna=False)
    .rename_axis("item_public")
    .reset_index(name="count")
)

print("Distribution by item_deleted:")
display(
    df_total_oai_records["item_deleted"]
    .value_counts(dropna=False)
    .rename_axis("item_deleted")
    .reset_index(name="count")
)

display(df_total_oai_records.head())

print(f"File saved to: {output_total_oai_records.resolve()}")

### Total OAI records in the reference month

In [ ]:
# =========================================================
# TOTAL OAI RECORDS IN THE REFERENCE MONTH
# =========================================================

# - item_id
# - item_handle
# - oai_record_id
# - item_accessioned_date
# - item_available_date
# - resource_issued_date
# - item_last_modified
# - item_last_modified_datetime
# - item_deleted
# - item_public
# - item_collections
# - item_communities
# - title
# - authors
# - resource_type
# - language
# - mimetype
# - core_name
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_value(value):
    """
    Normalizes Solr scalar/list values for CSV export.
    """

    if isinstance(value, list):
        return "; ".join(str(v) for v in value)

    return value


reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01", utc=True)
reference_month_end = reference_month_start + pd.DateOffset(months=1)

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
oai_record_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_last_modified = doc.get("item.lastmodified")

        oai_record_rows.append({
            "item_id": doc.get("item.id"),
            "item_handle": doc.get("item.handle"),
            "oai_record_id": doc.get("id"),
            "item_accessioned_date": normalize_solr_value(
                doc.get("metadata.dc.date.accessioned")
            ),
            "item_available_date": normalize_solr_value(
                doc.get("metadata.dc.date.available")
            ),
            "resource_issued_date": normalize_solr_value(
                doc.get("metadata.dc.date.issued")
            ),
            "item_last_modified": item_last_modified,
            "item_last_modified_datetime": pd.to_datetime(
                item_last_modified,
                errors="coerce",
                utc=True
            ),
            "item_deleted": doc.get("item.deleted"),
            "item_public": doc.get("item.public"),
            "item_collections": normalize_solr_value(
                doc.get("item.collections")
            ),
            "item_communities": normalize_solr_value(
                doc.get("item.communities")
            ),
            "title": normalize_solr_value(
                doc.get("metadata.dc.title")
            ),
            "authors": normalize_solr_value(
                doc.get("metadata.dc.contributor.author")
            ),
            "resource_type": normalize_solr_value(
                doc.get("metadata.dc.type")
            ),
            "language": normalize_solr_value(
                doc.get("metadata.dc.language.iso")
            ),
            "mimetype": normalize_solr_value(
                doc.get("metadata.dc.format.mimetype")
            ),
            "core_name": oai_core,
            "period_start": reference_month_start.strftime("%Y-%m-%d"),
            "period_end": reference_month_end.strftime("%Y-%m-%d"),
            "query": (
                "filtered in pandas using item.lastmodified "
                f"from {reference_month_start.strftime('%Y-%m-%d')} "
                f"to {reference_month_end.strftime('%Y-%m-%d')}"
            )
        })

df_total_oai_records_all_for_month_filter = pd.DataFrame(oai_record_rows)

df_total_oai_records_reference_month = (
    df_total_oai_records_all_for_month_filter[
        (
            df_total_oai_records_all_for_month_filter["item_last_modified_datetime"]
            >= reference_month_start
        )
        &
        (
            df_total_oai_records_all_for_month_filter["item_last_modified_datetime"]
            < reference_month_end
        )
    ]
    .copy()
)

df_total_oai_records_reference_month["reference_month"] = REPORT_MONTH
df_total_oai_records_reference_month["environment"] = ENVIRONMENT
df_total_oai_records_reference_month["source"] = "solr"
df_total_oai_records_reference_month["metric_definition"] = (
    "records indexed in the Solr OAI core and updated during the reference month, "
    "filtered in pandas using item.lastmodified; includes item_id, handle, "
    "accession/deposit date, availability date, issued date, last modification date, "
    "public/deleted status and selected descriptive metadata"
)

output_total_oai_records_reference_month = (
    export_path("solr_total_oai_records_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_total_oai_records_reference_month.to_csv(
    output_total_oai_records_reference_month,
    index=False
)

print(
    "Total OAI records in the reference month extracted: "
    f"{len(df_total_oai_records_reference_month)}"
)
print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

print("Distribution by item_public:")
display(
    df_total_oai_records_reference_month["item_public"]
    .value_counts(dropna=False)
    .rename_axis("item_public")
    .reset_index(name="count")
)

print("Distribution by item_deleted:")
display(
    df_total_oai_records_reference_month["item_deleted"]
    .value_counts(dropna=False)
    .rename_axis("item_deleted")
    .reset_index(name="count")
)

display(df_total_oai_records_reference_month.head())

print(f"File saved to: {output_total_oai_records_reference_month.resolve()}")

### First 10 OAI datestamp

In [ ]:
# =========================================================
# FIRST 10 OAI DATESTAMP
# =========================================================

# - item_id
# - item_handle
# - oai_record_id
# - oai_datestamp
# - oai_datestamp_datetime
# - item_accessioned_date
# - item_available_date
# - resource_issued_date
# - item_deleted
# - item_public
# - title
# - core_name
# - query
# - reference_month
# - environment
# - source
# - metric_definition

oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_value(value):
    """
    Normalizes Solr scalar/list values for CSV export.
    """

    if isinstance(value, list):
        return "; ".join(str(v) for v in value)

    return value


response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
oai_datestamp_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        oai_datestamp = doc.get("item.lastmodified")

        oai_datestamp_rows.append({
            "item_id": doc.get("item.id"),
            "item_handle": doc.get("item.handle"),
            "oai_record_id": doc.get("id"),
            "oai_datestamp": oai_datestamp,
            "oai_datestamp_datetime": pd.to_datetime(
                oai_datestamp,
                errors="coerce",
                utc=True
            ),
            "item_accessioned_date": normalize_solr_value(
                doc.get("metadata.dc.date.accessioned")
            ),
            "item_available_date": normalize_solr_value(
                doc.get("metadata.dc.date.available")
            ),
            "resource_issued_date": normalize_solr_value(
                doc.get("metadata.dc.date.issued")
            ),
            "item_deleted": doc.get("item.deleted"),
            "item_public": doc.get("item.public"),
            "title": normalize_solr_value(doc.get("metadata.dc.title")),
            "core_name": oai_core,
            "query": "*:*"
        })

df_oai_datestamps = pd.DataFrame(oai_datestamp_rows)

df_first_10_oai_datestamp = (
    df_oai_datestamps
    .dropna(subset=["oai_datestamp_datetime"])
    .sort_values("oai_datestamp_datetime", ascending=True)
    .head(10)
    .copy()
)

df_first_10_oai_datestamp["reference_month"] = REPORT_MONTH
df_first_10_oai_datestamp["environment"] = ENVIRONMENT
df_first_10_oai_datestamp["source"] = "solr"
df_first_10_oai_datestamp["metric_definition"] = (
    "first 10 OAI records by OAI datestamp, using item.lastmodified as the OAI datestamp"
)

output_first_10_oai_datestamp = export_path("solr_first_10_oai_datestamp.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_first_10_oai_datestamp.to_csv(output_first_10_oai_datestamp, index=False)

print(f"First 10 OAI datestamp records extracted: {len(df_first_10_oai_datestamp)}")

display(df_first_10_oai_datestamp)

print(f"File saved to: {output_first_10_oai_datestamp.resolve()}")

### Last 10 OAI datestamp

In [ ]:
# =========================================================
# LAST 10 OAI DATESTAMP
# =========================================================

# - item_id
# - item_handle
# - oai_record_id
# - oai_datestamp
# - oai_datestamp_datetime
# - item_accessioned_date
# - item_available_date
# - resource_issued_date
# - item_deleted
# - item_public
# - title
# - core_name
# - query
# - reference_month
# - environment
# - source
# - metric_definition

oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_value(value):
    """
    Normalizes Solr scalar/list values for CSV export.
    """

    if isinstance(value, list):
        return "; ".join(str(v) for v in value)

    return value


response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
oai_datestamp_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        oai_datestamp = doc.get("item.lastmodified")

        oai_datestamp_rows.append({
            "item_id": doc.get("item.id"),
            "item_handle": doc.get("item.handle"),
            "oai_record_id": doc.get("id"),
            "oai_datestamp": oai_datestamp,
            "oai_datestamp_datetime": pd.to_datetime(
                oai_datestamp,
                errors="coerce",
                utc=True
            ),
            "item_accessioned_date": normalize_solr_value(
                doc.get("metadata.dc.date.accessioned")
            ),
            "item_available_date": normalize_solr_value(
                doc.get("metadata.dc.date.available")
            ),
            "resource_issued_date": normalize_solr_value(
                doc.get("metadata.dc.date.issued")
            ),
            "item_deleted": doc.get("item.deleted"),
            "item_public": doc.get("item.public"),
            "title": normalize_solr_value(doc.get("metadata.dc.title")),
            "core_name": oai_core,
            "query": "*:*"
        })

df_oai_datestamps = pd.DataFrame(oai_datestamp_rows)

df_last_10_oai_datestamp = (
    df_oai_datestamps
    .dropna(subset=["oai_datestamp_datetime"])
    .sort_values("oai_datestamp_datetime", ascending=False)
    .head(10)
    .copy()
)

df_last_10_oai_datestamp["reference_month"] = REPORT_MONTH
df_last_10_oai_datestamp["environment"] = ENVIRONMENT
df_last_10_oai_datestamp["source"] = "solr"
df_last_10_oai_datestamp["metric_definition"] = (
    "last 10 OAI records by OAI datestamp, using item.lastmodified as the OAI datestamp"
)

output_last_10_oai_datestamp = export_path("solr_last_10_oai_datestamp.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_last_10_oai_datestamp.to_csv(output_last_10_oai_datestamp, index=False)

print(f"Last 10 OAI datestamp records extracted: {len(df_last_10_oai_datestamp)}")

display(df_last_10_oai_datestamp)

print(f"File saved to: {output_last_10_oai_datestamp.resolve()}")

### OAI records with missing informations

In [ ]:
# =========================================================
# OAI RECORDS WITH MISSING OR EMPTY INFORMATION
# =========================================================

# - item_id
# - item_handle
# - oai_record_id
# - missing_fields
# - missing_fields_count
# - item_accessioned_date
# - item_available_date
# - resource_issued_date
# - item_last_modified
# - item_deleted
# - item_public
# - title
# - authors
# - resource_type
# - language
# - mimetype
# - core_name
# - query
# - reference_month
# - environment
# - source
# - metric_definition

oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_value(value):
    """
    Normalizes Solr scalar/list values for CSV export.
    """

    if isinstance(value, list):
        return "; ".join(str(v) for v in value)

    return value


def is_missing_or_empty(value):
    """
    Checks whether a Solr value is missing, empty, or contains only empty values.
    """

    if value is None:
        return True

    if isinstance(value, list):
        return len(value) == 0 or all(
            v is None or str(v).strip() == ""
            for v in value
        )

    return str(value).strip() == ""


required_oai_fields = {
    "item_id": "item.id",
    "item_handle": "item.handle",
    "item_accessioned_date": "metadata.dc.date.accessioned",
    "item_last_modified": "item.lastmodified",
    "item_public": "item.public",
    "title": "metadata.dc.title"
}

optional_but_relevant_oai_fields = {
    "item_available_date": "metadata.dc.date.available",
    "resource_issued_date": "metadata.dc.date.issued",
    "authors": "metadata.dc.contributor.author",
    "resource_type": "metadata.dc.type",
    "language": "metadata.dc.language.iso",
    "mimetype": "metadata.dc.format.mimetype"
}

fields_to_check = {
    **required_oai_fields,
    **optional_but_relevant_oai_fields
}

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
oai_missing_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        missing_fields = []

        for logical_name, solr_field in fields_to_check.items():
            if is_missing_or_empty(doc.get(solr_field)):
                missing_fields.append(logical_name)

        if missing_fields:
            oai_missing_rows.append({
                "item_id": doc.get("item.id"),
                "item_handle": doc.get("item.handle"),
                "oai_record_id": doc.get("id"),
                "missing_fields": "; ".join(missing_fields),
                "missing_fields_count": len(missing_fields),
                "item_accessioned_date": normalize_solr_value(
                    doc.get("metadata.dc.date.accessioned")
                ),
                "item_available_date": normalize_solr_value(
                    doc.get("metadata.dc.date.available")
                ),
                "resource_issued_date": normalize_solr_value(
                    doc.get("metadata.dc.date.issued")
                ),
                "item_last_modified": doc.get("item.lastmodified"),
                "item_deleted": doc.get("item.deleted"),
                "item_public": doc.get("item.public"),
                "title": normalize_solr_value(
                    doc.get("metadata.dc.title")
                ),
                "authors": normalize_solr_value(
                    doc.get("metadata.dc.contributor.author")
                ),
                "resource_type": normalize_solr_value(
                    doc.get("metadata.dc.type")
                ),
                "language": normalize_solr_value(
                    doc.get("metadata.dc.language.iso")
                ),
                "mimetype": normalize_solr_value(
                    doc.get("metadata.dc.format.mimetype")
                ),
                "core_name": oai_core,
                "query": "*:*"
            })

df_oai_records_missing_information = pd.DataFrame(oai_missing_rows)

df_oai_records_missing_information["reference_month"] = REPORT_MONTH
df_oai_records_missing_information["environment"] = ENVIRONMENT
df_oai_records_missing_information["source"] = "solr"
df_oai_records_missing_information["metric_definition"] = (
    "OAI records with missing or empty relevant information in the Solr OAI core. "
    "The check includes item_id, handle, accession date, last modification date, "
    "public status, title, availability date, issued date, authors, resource type, "
    "language and MIME type."
)

output_oai_records_missing_information = (
    export_path("solr_oai_records_missing_information.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_oai_records_missing_information.to_csv(
    output_oai_records_missing_information,
    index=False
)

print(
    "OAI records with missing or empty information: "
    f"{len(df_oai_records_missing_information)}"
)

if not df_oai_records_missing_information.empty:
    print("Distribution by missing field:")
    missing_field_distribution = (
        df_oai_records_missing_information["missing_fields"]
        .str.split("; ")
        .explode()
        .value_counts(dropna=False)
        .rename_axis("missing_field")
        .reset_index(name="records_count")
    )

    display(missing_field_distribution)

display(df_oai_records_missing_information.head())

print(f"File saved to: {output_oai_records_missing_information.resolve()}")

### OAI records of deleted items

In [ ]:
# =========================================================
# OAI RECORDS OF DELETED ITEMS
# =========================================================

# - item_id
# - item_handle
# - oai_record_id
# - item_deleted
# - item_public
# - item_accessioned_date
# - item_available_date
# - resource_issued_date
# - item_last_modified
# - title
# - authors
# - resource_type
# - language
# - mimetype
# - core_name
# - query
# - reference_month
# - environment
# - source
# - metric_definition

oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_value(value):
    """
    Normalizes Solr scalar/list values for CSV export.
    """

    if isinstance(value, list):
        return "; ".join(str(v) for v in value)

    return value


response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
oai_deleted_item_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_deleted = doc.get("item.deleted")

        if item_deleted is True or str(item_deleted).lower() == "true":
            oai_deleted_item_rows.append({
                "item_id": doc.get("item.id"),
                "item_handle": doc.get("item.handle"),
                "oai_record_id": doc.get("id"),
                "item_deleted": item_deleted,
                "item_public": doc.get("item.public"),
                "item_accessioned_date": normalize_solr_value(
                    doc.get("metadata.dc.date.accessioned")
                ),
                "item_available_date": normalize_solr_value(
                    doc.get("metadata.dc.date.available")
                ),
                "resource_issued_date": normalize_solr_value(
                    doc.get("metadata.dc.date.issued")
                ),
                "item_last_modified": doc.get("item.lastmodified"),
                "title": normalize_solr_value(
                    doc.get("metadata.dc.title")
                ),
                "authors": normalize_solr_value(
                    doc.get("metadata.dc.contributor.author")
                ),
                "resource_type": normalize_solr_value(
                    doc.get("metadata.dc.type")
                ),
                "language": normalize_solr_value(
                    doc.get("metadata.dc.language.iso")
                ),
                "mimetype": normalize_solr_value(
                    doc.get("metadata.dc.format.mimetype")
                ),
                "core_name": oai_core,
                "query": "item.deleted:true"
            })

df_oai_records_deleted_items = pd.DataFrame(oai_deleted_item_rows)

df_oai_records_deleted_items["reference_month"] = REPORT_MONTH
df_oai_records_deleted_items["environment"] = ENVIRONMENT
df_oai_records_deleted_items["source"] = "solr"
df_oai_records_deleted_items["metric_definition"] = (
    "OAI records associated with items marked as deleted in the Solr OAI core, "
    "based on the item.deleted field"
)

output_oai_records_deleted_items = (
    export_path("solr_oai_records_deleted_items.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_oai_records_deleted_items.to_csv(
    output_oai_records_deleted_items,
    index=False
)

print(
    "OAI records of deleted items: "
    f"{len(df_oai_records_deleted_items)}"
)

if not df_oai_records_deleted_items.empty:
    print("Distribution by item_public:")
    display(
        df_oai_records_deleted_items["item_public"]
        .value_counts(dropna=False)
        .rename_axis("item_public")
        .reset_index(name="count")
    )

display(df_oai_records_deleted_items.head())

print(f"File saved to: {output_oai_records_deleted_items.resolve()}")

## **Integrated metrics**

### Views by collection

In [ ]:
# =========================================================
# VIEWS BY COLLECTION
# =========================================================

# - collection_id
# - views_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bot_filter
# - facet_field
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_views_by_collection = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:3",
            "statistics_type:view",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_views_by_collection
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

views_by_collection_rows = []

for i in range(0, len(facet_values), 2):
    views_by_collection_rows.append({
        "collection_id": facet_values[i],
        "views_count": facet_values[i + 1],
        "core_name": statistics_core,
        "object_type": "3",
        "object_type_label": "collection",
        "statistics_type": "view",
        "bot_filter": "excluded",
        "facet_field": "id",
        "query": "type:3 AND statistics_type:view AND -isBot:true FACET id"
    })

df_views_by_collection = pd.DataFrame(views_by_collection_rows)

df_views_by_collection["reference_month"] = REPORT_MONTH
df_views_by_collection["environment"] = ENVIRONMENT
df_views_by_collection["source"] = "solr"
df_views_by_collection["metric_definition"] = (
    "collection view events recorded in the Solr statistics core, grouped by collection id "
    "and excluding bot events"
)

output_views_by_collection = export_path("solr_views_by_collection.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_views_by_collection.to_csv(output_views_by_collection, index=False)

print(f"Collections with views: {len(df_views_by_collection)}")
print(f"Total collection views excluding bots: {df_views_by_collection['views_count'].sum() if not df_views_by_collection.empty else 0}")

display(df_views_by_collection.head())

print(f"File saved to: {output_views_by_collection.resolve()}")

### Views by collection in the reference month

In [ ]:
# =========================================================
# VIEWS BY COLLECTION IN THE REFERENCE MONTH
# =========================================================

# - collection_id
# - views_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bot_filter
# - facet_field
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

response_views_by_collection_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:3",
            "statistics_type:view",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_views_by_collection_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

views_by_collection_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    views_by_collection_reference_month_rows.append({
        "collection_id": facet_values[i],
        "views_count": facet_values[i + 1],
        "core_name": statistics_core,
        "object_type": "3",
        "object_type_label": "collection",
        "statistics_type": "view",
        "bot_filter": "excluded",
        "facet_field": "id",
        "period_start": reference_month_start.strftime("%Y-%m-%d"),
        "period_end": reference_month_end.strftime("%Y-%m-%d"),
        "query": (
            "type:3 AND statistics_type:view AND -isBot:true "
            f"AND {time_filter} FACET id"
        )
    })

df_views_by_collection_reference_month = pd.DataFrame(
    views_by_collection_reference_month_rows
)

df_views_by_collection_reference_month["reference_month"] = REPORT_MONTH
df_views_by_collection_reference_month["environment"] = ENVIRONMENT
df_views_by_collection_reference_month["source"] = "solr"
df_views_by_collection_reference_month["metric_definition"] = (
    "collection view events recorded in the Solr statistics core during the reference month, "
    "grouped by collection id and excluding bot events"
)

output_views_by_collection_reference_month = (
    export_path("solr_views_by_collection_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_views_by_collection_reference_month.to_csv(
    output_views_by_collection_reference_month,
    index=False
)

print(
    "Collections with views in the reference month: "
    f"{len(df_views_by_collection_reference_month)}"
)

print(
    "Total collection views in the reference month excluding bots: "
    f"{df_views_by_collection_reference_month['views_count'].sum() if not df_views_by_collection_reference_month.empty else 0}"
)

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_views_by_collection_reference_month.head())

print(f"File saved to: {output_views_by_collection_reference_month.resolve()}")

### Downloads by collection

In [ ]:
# =========================================================
# DOWNLOADS BY COLLECTION
# =========================================================

# - collection_id
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [str(v) for v in value if v is not None and str(v).strip() != ""]

    if str(value).strip() == "":
        return []

    return [str(value)]


# =========================================================
# STEP 1: BUILD ITEM -> COLLECTIONS MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_collection_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")
        collection_ids = normalize_solr_list(doc.get("item.collections"))

        for collection_id in collection_ids:
            item_collection_rows.append({
                "item_id": item_id,
                "collection_id": collection_id
            })

df_item_collections_from_oai = pd.DataFrame(item_collection_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item = pd.DataFrame(downloads_by_item_rows)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> COLLECTIONS MAP
# =========================================================

if not df_downloads_by_item.empty and not df_item_collections_from_oai.empty:
    df_downloads_by_item_collection = df_downloads_by_item.merge(
        df_item_collections_from_oai,
        on="item_id",
        how="left"
    )

    df_downloads_by_collection = (
        df_downloads_by_item_collection
        .groupby("collection_id", dropna=False)
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=("item_id", lambda x: "; ".join(sorted(set(x.dropna().astype(str))))),
            bitstream_ids=("bitstream_ids", lambda x: "; ".join(sorted(set("; ".join(x.dropna().astype(str)).split("; ")))))
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_collection = pd.DataFrame(
        columns=[
            "collection_id",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_collection["core_name"] = statistics_core
df_downloads_by_collection["statistics_core"] = statistics_core
df_downloads_by_collection["oai_core"] = oai_core
df_downloads_by_collection["object_type"] = "0"
df_downloads_by_collection["object_type_label"] = "bitstream"
df_downloads_by_collection["statistics_type"] = "view"
df_downloads_by_collection["bundle_name"] = "ORIGINAL"
df_downloads_by_collection["bot_filter"] = "excluded"
df_downloads_by_collection["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    "AND -isBot:true FACET owningItem; OAI core used for item.id -> item.collections mapping"
)

df_downloads_by_collection["reference_month"] = REPORT_MONTH
df_downloads_by_collection["environment"] = ENVIRONMENT
df_downloads_by_collection["source"] = "solr"
df_downloads_by_collection["metric_definition"] = (
    "bitstream downloads grouped by collection. Downloads are extracted from the Solr "
    "statistics core and aggregated by owningItem; collection membership is obtained "
    "from the Solr OAI core using item.id and item.collections. Bot events are excluded."
)

output_downloads_by_collection = export_path("solr_downloads_by_collection.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_collection.to_csv(output_downloads_by_collection, index=False)

print(f"Collections with downloads: {len(df_downloads_by_collection)}")
print(
    "Total downloads by collection excluding bots: "
    f"{df_downloads_by_collection['downloads_count'].sum() if not df_downloads_by_collection.empty else 0}"
)

display(df_downloads_by_collection.head())

print(f"File saved to: {output_downloads_by_collection.resolve()}")

### Downloads by collection in the reference month

In [ ]:
# =========================================================
# VIEWS BY COMMUNITY
# =========================================================

# - community_id
# - community_name
# - views_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bot_filter
# - facet_field
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

community_labels = {
    "bdf96608-5c94-471c-9996-63942e832141": "ILC",
    "dc62d695-9dbb-4b9d-baa5-a49ab45288cd": "OPEN"
}

response_views_by_community = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:4",
            "statistics_type:view",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_views_by_community
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

views_by_community_rows = []

for i in range(0, len(facet_values), 2):
    community_id = facet_values[i]

    views_by_community_rows.append({
        "community_id": community_id,
        "community_name": community_labels.get(
            community_id,
            "unknown_or_historical"
        ),
        "views_count": facet_values[i + 1],
        "core_name": statistics_core,
        "object_type": "4",
        "object_type_label": "community",
        "statistics_type": "view",
        "bot_filter": "excluded",
        "facet_field": "id",
        "query": "type:4 AND statistics_type:view AND -isBot:true FACET id"
    })

df_views_by_community = pd.DataFrame(views_by_community_rows)

df_views_by_community["reference_month"] = REPORT_MONTH
df_views_by_community["environment"] = ENVIRONMENT
df_views_by_community["source"] = "solr"
df_views_by_community["metric_definition"] = (
    "community view events recorded in the Solr statistics core, grouped by community id "
    "and excluding bot events; current community names are mapped for ILC and OPEN, "
    "while other UUIDs are marked as unknown_or_historical"
)

output_views_by_community = export_path("solr_views_by_community.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_views_by_community.to_csv(output_views_by_community, index=False)

print(f"Communities with views: {len(df_views_by_community)}")
print(
    "Total community views excluding bots: "
    f"{df_views_by_community['views_count'].sum() if not df_views_by_community.empty else 0}"
)

print("Distribution by community name:")
display(
    df_views_by_community
    .groupby("community_name", dropna=False)["views_count"]
    .sum()
    .reset_index()
    .sort_values("views_count", ascending=False)
)

display(df_views_by_community.head())

print(f"File saved to: {output_views_by_community.resolve()}")

### Views by community

In [ ]:
# =========================================================
# VIEWS BY COMMUNITY
# =========================================================

# - community_id
# - views_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bot_filter
# - facet_field
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

response_views_by_community = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:4",
            "statistics_type:view",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_views_by_community
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

views_by_community_rows = []

for i in range(0, len(facet_values), 2):
    views_by_community_rows.append({
        "community_id": facet_values[i],
        "views_count": facet_values[i + 1],
        "core_name": statistics_core,
        "object_type": "4",
        "object_type_label": "community",
        "statistics_type": "view",
        "bot_filter": "excluded",
        "facet_field": "id",
        "query": "type:4 AND statistics_type:view AND -isBot:true FACET id"
    })

df_views_by_community = pd.DataFrame(views_by_community_rows)

df_views_by_community["reference_month"] = REPORT_MONTH
df_views_by_community["environment"] = ENVIRONMENT
df_views_by_community["source"] = "solr"
df_views_by_community["metric_definition"] = (
    "community view events recorded in the Solr statistics core, grouped by community id "
    "and excluding bot events"
)

output_views_by_community = export_path("solr_views_by_community.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_views_by_community.to_csv(output_views_by_community, index=False)

print(f"Communities with views: {len(df_views_by_community)}")
print(
    "Total community views excluding bots: "
    f"{df_views_by_community['views_count'].sum() if not df_views_by_community.empty else 0}"
)

display(df_views_by_community.head())

print(f"File saved to: {output_views_by_community.resolve()}")

### Views by community in the reference month

In [ ]:
# =========================================================
# VIEWS BY CURRENT COMMUNITY IN THE REFERENCE MONTH
# =========================================================

# - community_id
# - community_name
# - views_count
# - core_name
# - object_type
# - object_type_label
# - statistics_type
# - bot_filter
# - facet_field
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"

current_community_labels = {
    "bdf96608-5c94-471c-9996-63942e832141": "ILC",
    "dc62d695-9dbb-4b9d-baa5-a49ab45288cd": "OPEN"
}

current_community_filter = (
    "id:("
    + " OR ".join(f'"{community_id}"' for community_id in current_community_labels.keys())
    + ")"
)

response_views_by_current_community_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:4",
            "statistics_type:view",
            "-isBot:true",
            time_filter,
            current_community_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_views_by_current_community_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

views_by_current_community_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    community_id = facet_values[i]

    views_by_current_community_reference_month_rows.append({
        "community_id": community_id,
        "community_name": current_community_labels.get(
            community_id,
            "unknown_or_historical"
        ),
        "views_count": facet_values[i + 1],
        "core_name": statistics_core,
        "object_type": "4",
        "object_type_label": "community",
        "statistics_type": "view",
        "bot_filter": "excluded",
        "facet_field": "id",
        "period_start": reference_month_start.strftime("%Y-%m-%d"),
        "period_end": reference_month_end.strftime("%Y-%m-%d"),
        "query": (
            "type:4 AND statistics_type:view AND -isBot:true "
            f"AND {time_filter} "
            "AND current community IDs only "
            "FACET id"
        )
    })

df_views_by_current_community_reference_month = pd.DataFrame(
    views_by_current_community_reference_month_rows,
    columns=[
        "community_id",
        "community_name",
        "views_count",
        "core_name",
        "object_type",
        "object_type_label",
        "statistics_type",
        "bot_filter",
        "facet_field",
        "period_start",
        "period_end",
        "query",
    ],
)

# Ensure both current communities are always present, even if they have zero views.
df_current_communities = pd.DataFrame([
    {
        "community_id": community_id,
        "community_name": community_name
    }
    for community_id, community_name in current_community_labels.items()
])

df_views_by_current_community_reference_month = (
    df_current_communities
    .merge(
        df_views_by_current_community_reference_month,
        on=["community_id", "community_name"],
        how="left"
    )
)

df_views_by_current_community_reference_month["views_count"] = (
    df_views_by_current_community_reference_month["views_count"]
    .fillna(0)
    .astype(int)
)

df_views_by_current_community_reference_month["core_name"] = (
    df_views_by_current_community_reference_month["core_name"]
    .fillna(statistics_core)
)

df_views_by_current_community_reference_month["object_type"] = (
    df_views_by_current_community_reference_month["object_type"]
    .fillna("4")
)

df_views_by_current_community_reference_month["object_type_label"] = (
    df_views_by_current_community_reference_month["object_type_label"]
    .fillna("community")
)

df_views_by_current_community_reference_month["statistics_type"] = (
    df_views_by_current_community_reference_month["statistics_type"]
    .fillna("view")
)

df_views_by_current_community_reference_month["bot_filter"] = (
    df_views_by_current_community_reference_month["bot_filter"]
    .fillna("excluded")
)

df_views_by_current_community_reference_month["facet_field"] = (
    df_views_by_current_community_reference_month["facet_field"]
    .fillna("id")
)

df_views_by_current_community_reference_month["period_start"] = (
    df_views_by_current_community_reference_month["period_start"]
    .fillna(reference_month_start.strftime("%Y-%m-%d"))
)

df_views_by_current_community_reference_month["period_end"] = (
    df_views_by_current_community_reference_month["period_end"]
    .fillna(reference_month_end.strftime("%Y-%m-%d"))
)

df_views_by_current_community_reference_month["query"] = (
    df_views_by_current_community_reference_month["query"]
    .fillna(
        "type:4 AND statistics_type:view AND -isBot:true "
        f"AND {time_filter} "
        "AND current community IDs only FACET id"
    )
)

df_views_by_current_community_reference_month["reference_month"] = REPORT_MONTH
df_views_by_current_community_reference_month["environment"] = ENVIRONMENT
df_views_by_current_community_reference_month["source"] = "solr"
df_views_by_current_community_reference_month["metric_definition"] = (
    "community view events recorded in the Solr statistics core during the reference month, "
    "restricted to the two current public communities, ILC and OPEN, and excluding bot events"
)

df_views_by_current_community_reference_month = (
    df_views_by_current_community_reference_month
    .sort_values("views_count", ascending=False)
)

output_views_by_current_community_reference_month = (
    export_path("solr_views_by_current_community_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_views_by_current_community_reference_month.to_csv(
    output_views_by_current_community_reference_month,
    index=False
)

print(
    "Current communities with views in the reference month: "
    f"{len(df_views_by_current_community_reference_month)}"
)

print(
    "Total current community views in the reference month excluding bots: "
    f"{df_views_by_current_community_reference_month['views_count'].sum()}"
)

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_views_by_current_community_reference_month)

print(f"File saved to: {output_views_by_current_community_reference_month.resolve()}")

### Downloads by community

In [ ]:
# =========================================================
# DOWNLOADS BY COMMUNITY
# =========================================================

# - community_id
# - community_name
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")

community_labels = {
    "bdf96608-5c94-471c-9996-63942e832141": "ILC",
    "dc62d695-9dbb-4b9d-baa5-a49ab45288cd": "OPEN"
}


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(v)
            for v in value
            if v is not None and str(v).strip() != ""
        ]

    if str(value).strip() == "":
        return []

    return [str(value)]


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: BUILD ITEM -> COMMUNITIES MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_community_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")
        community_ids = normalize_solr_list(doc.get("item.communities"))

        for community_id in community_ids:
            item_community_rows.append({
                "item_id": item_id,
                "community_id": community_id,
                "community_name": community_labels.get(
                    community_id,
                    "unknown_or_historical"
                )
            })

df_item_communities_from_oai = pd.DataFrame(item_community_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item = pd.DataFrame(downloads_by_item_rows)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> COMMUNITIES MAP
# =========================================================

if not df_downloads_by_item.empty and not df_item_communities_from_oai.empty:
    df_downloads_by_item_community = df_downloads_by_item.merge(
        df_item_communities_from_oai,
        on="item_id",
        how="left"
    )

    df_downloads_by_community = (
        df_downloads_by_item_community
        .groupby(
            ["community_id", "community_name"],
            dropna=False
        )
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=(
                "item_id",
                lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
            ),
            bitstream_ids=("bitstream_ids", join_unique_values)
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_community = pd.DataFrame(
        columns=[
            "community_id",
            "community_name",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_community["core_name"] = statistics_core
df_downloads_by_community["statistics_core"] = statistics_core
df_downloads_by_community["oai_core"] = oai_core
df_downloads_by_community["object_type"] = "0"
df_downloads_by_community["object_type_label"] = "bitstream"
df_downloads_by_community["statistics_type"] = "view"
df_downloads_by_community["bundle_name"] = "ORIGINAL"
df_downloads_by_community["bot_filter"] = "excluded"
df_downloads_by_community["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    "AND -isBot:true FACET owningItem; OAI core used for item.id -> item.communities mapping"
)

df_downloads_by_community["reference_month"] = REPORT_MONTH
df_downloads_by_community["environment"] = ENVIRONMENT
df_downloads_by_community["source"] = "solr"
df_downloads_by_community["metric_definition"] = (
    "bitstream downloads grouped by community. Downloads are extracted from the Solr "
    "statistics core and aggregated by owningItem; community membership is obtained "
    "from the Solr OAI core using item.id and item.communities. Bot events are excluded. "
    "Current community names are mapped for ILC and OPEN; other UUIDs are marked as "
    "unknown_or_historical."
)

output_downloads_by_community = export_path("solr_downloads_by_community.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_community.to_csv(output_downloads_by_community, index=False)

print(f"Communities with downloads: {len(df_downloads_by_community)}")
print(
    "Total downloads by community excluding bots: "
    f"{df_downloads_by_community['downloads_count'].sum() if not df_downloads_by_community.empty else 0}"
)

print("Distribution by community name:")
display(
    df_downloads_by_community
    .groupby("community_name", dropna=False)["downloads_count"]
    .sum()
    .reset_index()
    .sort_values("downloads_count", ascending=False)
)

display(df_downloads_by_community.head())

print(f"File saved to: {output_downloads_by_community.resolve()}")

### Downloads by community in the reference month

In [ ]:
# =========================================================
# DOWNLOADS BY COMMUNITY IN THE REFERENCE MONTH
# =========================================================

# - community_id
# - community_name
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")

community_labels = {
    "bdf96608-5c94-471c-9996-63942e832141": "ILC",
    "dc62d695-9dbb-4b9d-baa5-a49ab45288cd": "OPEN"
}

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(v)
            for v in value
            if v is not None and str(v).strip() != ""
        ]

    if str(value).strip() == "":
        return []

    return [str(value)]


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: BUILD ITEM -> COMMUNITIES MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_community_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")
        community_ids = normalize_solr_list(doc.get("item.communities"))

        for community_id in community_ids:
            item_community_rows.append({
                "item_id": item_id,
                "community_id": community_id,
                "community_name": community_labels.get(
                    community_id,
                    "unknown_or_historical"
                )
            })

df_item_communities_from_oai = pd.DataFrame(item_community_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                time_filter,
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams_reference_month
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_reference_month_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item_reference_month = pd.DataFrame(
    downloads_by_item_reference_month_rows
)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> COMMUNITIES MAP
# =========================================================

if (
    not df_downloads_by_item_reference_month.empty
    and not df_item_communities_from_oai.empty
):
    df_downloads_by_item_community_reference_month = (
        df_downloads_by_item_reference_month
        .merge(
            df_item_communities_from_oai,
            on="item_id",
            how="left"
        )
    )

    df_downloads_by_community_reference_month = (
        df_downloads_by_item_community_reference_month
        .groupby(
            ["community_id", "community_name"],
            dropna=False
        )
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=(
                "item_id",
                lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
            ),
            bitstream_ids=("bitstream_ids", join_unique_values)
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_community_reference_month = pd.DataFrame(
        columns=[
            "community_id",
            "community_name",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_community_reference_month["core_name"] = statistics_core
df_downloads_by_community_reference_month["statistics_core"] = statistics_core
df_downloads_by_community_reference_month["oai_core"] = oai_core
df_downloads_by_community_reference_month["object_type"] = "0"
df_downloads_by_community_reference_month["object_type_label"] = "bitstream"
df_downloads_by_community_reference_month["statistics_type"] = "view"
df_downloads_by_community_reference_month["bundle_name"] = "ORIGINAL"
df_downloads_by_community_reference_month["bot_filter"] = "excluded"
df_downloads_by_community_reference_month["period_start"] = (
    reference_month_start.strftime("%Y-%m-%d")
)
df_downloads_by_community_reference_month["period_end"] = (
    reference_month_end.strftime("%Y-%m-%d")
)
df_downloads_by_community_reference_month["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    f"AND -isBot:true AND {time_filter} FACET owningItem; "
    "OAI core used for item.id -> item.communities mapping"
)

df_downloads_by_community_reference_month["reference_month"] = REPORT_MONTH
df_downloads_by_community_reference_month["environment"] = ENVIRONMENT
df_downloads_by_community_reference_month["source"] = "solr"
df_downloads_by_community_reference_month["metric_definition"] = (
    "bitstream downloads during the reference month grouped by community. Downloads "
    "are extracted from the Solr statistics core and aggregated by owningItem; community "
    "membership is obtained from the Solr OAI core using item.id and item.communities. "
    "Bot events are excluded. Current community names are mapped for ILC and OPEN; "
    "other UUIDs are marked as unknown_or_historical."
)

output_downloads_by_community_reference_month = (
    export_path("solr_downloads_by_community_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_community_reference_month.to_csv(
    output_downloads_by_community_reference_month,
    index=False
)

print(
    "Communities with downloads in the reference month: "
    f"{len(df_downloads_by_community_reference_month)}"
)

print(
    "Total downloads by community in the reference month excluding bots: "
    f"{df_downloads_by_community_reference_month['downloads_count'].sum() if not df_downloads_by_community_reference_month.empty else 0}"
)

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

print("Distribution by community name:")
display(
    df_downloads_by_community_reference_month
    .groupby("community_name", dropna=False)["downloads_count"]
    .sum()
    .reset_index()
    .sort_values("downloads_count", ascending=False)
)

display(df_downloads_by_community_reference_month.head())

print(f"File saved to: {output_downloads_by_community_reference_month.resolve()}")

### Downloads by license

In [ ]:
# =========================================================
# DOWNLOADS BY LICENSE
# =========================================================

# - license_value
# - license_uri
# - license_label
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(v)
            for v in value
            if v is not None and str(v).strip() != ""
        ]

    if str(value).strip() == "":
        return []

    return [str(value)]


def normalize_solr_value(value):
    """
    Normalizes Solr scalar/list values for CSV export.
    """

    if isinstance(value, list):
        return "; ".join(str(v) for v in value)

    return value


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: BUILD ITEM -> LICENSE MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_license_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")

        license_values = normalize_solr_list(doc.get("metadata.dc.rights"))
        license_uris = normalize_solr_list(doc.get("metadata.dc.rights.uri"))
        license_labels = normalize_solr_list(doc.get("metadata.dc.rights.label"))

        if not license_values and not license_uris and not license_labels:
            item_license_rows.append({
                "item_id": item_id,
                "license_value": "missing_license",
                "license_uri": None,
                "license_label": None
            })
        else:
            max_len = max(
                len(license_values),
                len(license_uris),
                len(license_labels),
                1
            )

            for index in range(max_len):
                item_license_rows.append({
                    "item_id": item_id,
                    "license_value": (
                        license_values[index]
                        if index < len(license_values)
                        else None
                    ),
                    "license_uri": (
                        license_uris[index]
                        if index < len(license_uris)
                        else None
                    ),
                    "license_label": (
                        license_labels[index]
                        if index < len(license_labels)
                        else None
                    )
                })

df_item_licenses_from_oai = pd.DataFrame(item_license_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item = pd.DataFrame(downloads_by_item_rows)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> LICENSE MAP
# =========================================================

if not df_downloads_by_item.empty and not df_item_licenses_from_oai.empty:
    df_downloads_by_item_license = df_downloads_by_item.merge(
        df_item_licenses_from_oai,
        on="item_id",
        how="left"
    )

    df_downloads_by_item_license["license_value"] = (
        df_downloads_by_item_license["license_value"]
        .fillna("missing_license")
    )

    df_downloads_by_license = (
        df_downloads_by_item_license
        .groupby(
            ["license_value", "license_uri", "license_label"],
            dropna=False
        )
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=(
                "item_id",
                lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
            ),
            bitstream_ids=("bitstream_ids", join_unique_values)
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_license = pd.DataFrame(
        columns=[
            "license_value",
            "license_uri",
            "license_label",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_license["core_name"] = statistics_core
df_downloads_by_license["statistics_core"] = statistics_core
df_downloads_by_license["oai_core"] = oai_core
df_downloads_by_license["object_type"] = "0"
df_downloads_by_license["object_type_label"] = "bitstream"
df_downloads_by_license["statistics_type"] = "view"
df_downloads_by_license["bundle_name"] = "ORIGINAL"
df_downloads_by_license["bot_filter"] = "excluded"
df_downloads_by_license["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    "AND -isBot:true FACET owningItem; OAI core used for item.id -> metadata.dc.rights mapping"
)

df_downloads_by_license["reference_month"] = REPORT_MONTH
df_downloads_by_license["environment"] = ENVIRONMENT
df_downloads_by_license["source"] = "solr"
df_downloads_by_license["metric_definition"] = (
    "bitstream downloads grouped by license. Downloads are extracted from the Solr "
    "statistics core and aggregated by owningItem; license information is obtained "
    "from the Solr OAI core using item.id and metadata.dc.rights / metadata.dc.rights.uri / "
    "metadata.dc.rights.label. Bot events are excluded."
)

output_downloads_by_license = export_path("solr_downloads_by_license.csv")

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_license.to_csv(output_downloads_by_license, index=False)

print(f"License groups with downloads: {len(df_downloads_by_license)}")
print(
    "Total downloads by license excluding bots: "
    f"{df_downloads_by_license['downloads_count'].sum() if not df_downloads_by_license.empty else 0}"
)

display(df_downloads_by_license.head())

print(f"File saved to: {output_downloads_by_license.resolve()}")

### Downloads by license in the reference month

In [ ]:
# =========================================================
# DOWNLOADS BY LICENSE IN THE REFERENCE MONTH
# =========================================================

# - license_value
# - license_uri
# - license_label
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(v)
            for v in value
            if v is not None and str(v).strip() != ""
        ]

    if str(value).strip() == "":
        return []

    return [str(value)]


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: BUILD ITEM -> LICENSE MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_license_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")

        license_values = normalize_solr_list(doc.get("metadata.dc.rights"))
        license_uris = normalize_solr_list(doc.get("metadata.dc.rights.uri"))
        license_labels = normalize_solr_list(doc.get("metadata.dc.rights.label"))

        if not license_values and not license_uris and not license_labels:
            item_license_rows.append({
                "item_id": item_id,
                "license_value": "missing_license",
                "license_uri": None,
                "license_label": None
            })
        else:
            max_len = max(
                len(license_values),
                len(license_uris),
                len(license_labels),
                1
            )

            for index in range(max_len):
                item_license_rows.append({
                    "item_id": item_id,
                    "license_value": (
                        license_values[index]
                        if index < len(license_values)
                        else None
                    ),
                    "license_uri": (
                        license_uris[index]
                        if index < len(license_uris)
                        else None
                    ),
                    "license_label": (
                        license_labels[index]
                        if index < len(license_labels)
                        else None
                    )
                })

df_item_licenses_from_oai = pd.DataFrame(item_license_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                time_filter,
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams_reference_month
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_reference_month_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item_reference_month = pd.DataFrame(
    downloads_by_item_reference_month_rows
)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> LICENSE MAP
# =========================================================

if (
    not df_downloads_by_item_reference_month.empty
    and not df_item_licenses_from_oai.empty
):
    df_downloads_by_item_license_reference_month = (
        df_downloads_by_item_reference_month
        .merge(
            df_item_licenses_from_oai,
            on="item_id",
            how="left"
        )
    )

    df_downloads_by_item_license_reference_month["license_value"] = (
        df_downloads_by_item_license_reference_month["license_value"]
        .fillna("missing_license")
    )

    df_downloads_by_license_reference_month = (
        df_downloads_by_item_license_reference_month
        .groupby(
            ["license_value", "license_uri", "license_label"],
            dropna=False
        )
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=(
                "item_id",
                lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
            ),
            bitstream_ids=("bitstream_ids", join_unique_values)
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_license_reference_month = pd.DataFrame(
        columns=[
            "license_value",
            "license_uri",
            "license_label",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_license_reference_month["core_name"] = statistics_core
df_downloads_by_license_reference_month["statistics_core"] = statistics_core
df_downloads_by_license_reference_month["oai_core"] = oai_core
df_downloads_by_license_reference_month["object_type"] = "0"
df_downloads_by_license_reference_month["object_type_label"] = "bitstream"
df_downloads_by_license_reference_month["statistics_type"] = "view"
df_downloads_by_license_reference_month["bundle_name"] = "ORIGINAL"
df_downloads_by_license_reference_month["bot_filter"] = "excluded"
df_downloads_by_license_reference_month["period_start"] = (
    reference_month_start.strftime("%Y-%m-%d")
)
df_downloads_by_license_reference_month["period_end"] = (
    reference_month_end.strftime("%Y-%m-%d")
)
df_downloads_by_license_reference_month["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    f"AND -isBot:true AND {time_filter} FACET owningItem; "
    "OAI core used for item.id -> metadata.dc.rights mapping"
)

df_downloads_by_license_reference_month["reference_month"] = REPORT_MONTH
df_downloads_by_license_reference_month["environment"] = ENVIRONMENT
df_downloads_by_license_reference_month["source"] = "solr"
df_downloads_by_license_reference_month["metric_definition"] = (
    "bitstream downloads during the reference month grouped by license. Downloads are "
    "extracted from the Solr statistics core and aggregated by owningItem; license "
    "information is obtained from the Solr OAI core using item.id and metadata.dc.rights / "
    "metadata.dc.rights.uri / metadata.dc.rights.label. Bot events are excluded."
)

output_downloads_by_license_reference_month = (
    export_path("solr_downloads_by_license_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_license_reference_month.to_csv(
    output_downloads_by_license_reference_month,
    index=False
)

print(
    "License groups with downloads in the reference month: "
    f"{len(df_downloads_by_license_reference_month)}"
)

print(
    "Total downloads by license in the reference month excluding bots: "
    f"{df_downloads_by_license_reference_month['downloads_count'].sum() if not df_downloads_by_license_reference_month.empty else 0}"
)

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_downloads_by_license_reference_month.head())

print(f"File saved to: {output_downloads_by_license_reference_month.resolve()}")

### Downloads by resource type

In [ ]:
# =========================================================
# DOWNLOADS BY RESOURCE TYPE
# =========================================================

# - resource_type
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(v)
            for v in value
            if v is not None and str(v).strip() != ""
        ]

    if str(value).strip() == "":
        return []

    return [str(value)]


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: BUILD ITEM -> RESOURCE TYPE MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_resource_type_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")
        resource_types = normalize_solr_list(doc.get("metadata.dc.type"))

        if not resource_types:
            item_resource_type_rows.append({
                "item_id": item_id,
                "resource_type": "missing_resource_type"
            })
        else:
            for resource_type in resource_types:
                item_resource_type_rows.append({
                    "item_id": item_id,
                    "resource_type": resource_type
                })

df_item_resource_types_from_oai = pd.DataFrame(item_resource_type_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item = pd.DataFrame(downloads_by_item_rows)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> RESOURCE TYPE MAP
# =========================================================

if not df_downloads_by_item.empty and not df_item_resource_types_from_oai.empty:
    df_downloads_by_item_resource_type = df_downloads_by_item.merge(
        df_item_resource_types_from_oai,
        on="item_id",
        how="left"
    )

    df_downloads_by_item_resource_type["resource_type"] = (
        df_downloads_by_item_resource_type["resource_type"]
        .fillna("missing_resource_type")
    )

    df_downloads_by_resource_type = (
        df_downloads_by_item_resource_type
        .groupby("resource_type", dropna=False)
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=(
                "item_id",
                lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
            ),
            bitstream_ids=("bitstream_ids", join_unique_values)
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_resource_type = pd.DataFrame(
        columns=[
            "resource_type",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_resource_type["core_name"] = statistics_core
df_downloads_by_resource_type["statistics_core"] = statistics_core
df_downloads_by_resource_type["oai_core"] = oai_core
df_downloads_by_resource_type["object_type"] = "0"
df_downloads_by_resource_type["object_type_label"] = "bitstream"
df_downloads_by_resource_type["statistics_type"] = "view"
df_downloads_by_resource_type["bundle_name"] = "ORIGINAL"
df_downloads_by_resource_type["bot_filter"] = "excluded"
df_downloads_by_resource_type["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    "AND -isBot:true FACET owningItem; OAI core used for item.id -> metadata.dc.type mapping"
)

df_downloads_by_resource_type["reference_month"] = REPORT_MONTH
df_downloads_by_resource_type["environment"] = ENVIRONMENT
df_downloads_by_resource_type["source"] = "solr"
df_downloads_by_resource_type["metric_definition"] = (
    "bitstream downloads grouped by resource type. Downloads are extracted from the Solr "
    "statistics core and aggregated by owningItem; resource type information is obtained "
    "from the Solr OAI core using item.id and metadata.dc.type. Bot events are excluded."
)

output_downloads_by_resource_type = (
    export_path("solr_downloads_by_resource_type.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_resource_type.to_csv(
    output_downloads_by_resource_type,
    index=False
)

print(f"Resource type groups with downloads: {len(df_downloads_by_resource_type)}")
print(
    "Total downloads by resource type excluding bots: "
    f"{df_downloads_by_resource_type['downloads_count'].sum() if not df_downloads_by_resource_type.empty else 0}"
)

display(df_downloads_by_resource_type.head())

print(f"File saved to: {output_downloads_by_resource_type.resolve()}")

### Downloads by resource type in the reference month

In [ ]:
# =========================================================
# DOWNLOADS BY RESOURCE TYPE IN THE REFERENCE MONTH
# =========================================================

# - resource_type
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(v)
            for v in value
            if v is not None and str(v).strip() != ""
        ]

    if str(value).strip() == "":
        return []

    return [str(value)]


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: BUILD ITEM -> RESOURCE TYPE MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_resource_type_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")
        resource_types = normalize_solr_list(doc.get("metadata.dc.type"))

        if not resource_types:
            item_resource_type_rows.append({
                "item_id": item_id,
                "resource_type": "missing_resource_type"
            })
        else:
            for resource_type in resource_types:
                item_resource_type_rows.append({
                    "item_id": item_id,
                    "resource_type": resource_type
                })

df_item_resource_types_from_oai = pd.DataFrame(item_resource_type_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                time_filter,
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams_reference_month
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_reference_month_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item_reference_month = pd.DataFrame(
    downloads_by_item_reference_month_rows
)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> RESOURCE TYPE MAP
# =========================================================

if (
    not df_downloads_by_item_reference_month.empty
    and not df_item_resource_types_from_oai.empty
):
    df_downloads_by_item_resource_type_reference_month = (
        df_downloads_by_item_reference_month
        .merge(
            df_item_resource_types_from_oai,
            on="item_id",
            how="left"
        )
    )

    df_downloads_by_item_resource_type_reference_month["resource_type"] = (
        df_downloads_by_item_resource_type_reference_month["resource_type"]
        .fillna("missing_resource_type")
    )

    df_downloads_by_resource_type_reference_month = (
        df_downloads_by_item_resource_type_reference_month
        .groupby("resource_type", dropna=False)
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=(
                "item_id",
                lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
            ),
            bitstream_ids=("bitstream_ids", join_unique_values)
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_resource_type_reference_month = pd.DataFrame(
        columns=[
            "resource_type",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_resource_type_reference_month["core_name"] = statistics_core
df_downloads_by_resource_type_reference_month["statistics_core"] = statistics_core
df_downloads_by_resource_type_reference_month["oai_core"] = oai_core
df_downloads_by_resource_type_reference_month["object_type"] = "0"
df_downloads_by_resource_type_reference_month["object_type_label"] = "bitstream"
df_downloads_by_resource_type_reference_month["statistics_type"] = "view"
df_downloads_by_resource_type_reference_month["bundle_name"] = "ORIGINAL"
df_downloads_by_resource_type_reference_month["bot_filter"] = "excluded"
df_downloads_by_resource_type_reference_month["period_start"] = (
    reference_month_start.strftime("%Y-%m-%d")
)
df_downloads_by_resource_type_reference_month["period_end"] = (
    reference_month_end.strftime("%Y-%m-%d")
)
df_downloads_by_resource_type_reference_month["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    f"AND -isBot:true AND {time_filter} FACET owningItem; "
    "OAI core used for item.id -> metadata.dc.type mapping"
)

df_downloads_by_resource_type_reference_month["reference_month"] = REPORT_MONTH
df_downloads_by_resource_type_reference_month["environment"] = ENVIRONMENT
df_downloads_by_resource_type_reference_month["source"] = "solr"
df_downloads_by_resource_type_reference_month["metric_definition"] = (
    "bitstream downloads during the reference month grouped by resource type. Downloads "
    "are extracted from the Solr statistics core and aggregated by owningItem; resource "
    "type information is obtained from the Solr OAI core using item.id and metadata.dc.type. "
    "Bot events are excluded."
)

output_downloads_by_resource_type_reference_month = (
    export_path("solr_downloads_by_resource_type_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_resource_type_reference_month.to_csv(
    output_downloads_by_resource_type_reference_month,
    index=False
)

print(
    "Resource type groups with downloads in the reference month: "
    f"{len(df_downloads_by_resource_type_reference_month)}"
)

print(
    "Total downloads by resource type in the reference month excluding bots: "
    f"{df_downloads_by_resource_type_reference_month['downloads_count'].sum() if not df_downloads_by_resource_type_reference_month.empty else 0}"
)

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_downloads_by_resource_type_reference_month.head())

print(f"File saved to: {output_downloads_by_resource_type_reference_month.resolve()}")

### Downloads by MIME type

In [ ]:
# =========================================================
# DOWNLOADS BY MIME TYPE
# =========================================================

# - mimetype
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(v)
            for v in value
            if v is not None and str(v).strip() != ""
        ]

    if str(value).strip() == "":
        return []

    return [str(value)]


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: BUILD ITEM -> MIME TYPE MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_mimetype_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")
        mimetypes = normalize_solr_list(doc.get("metadata.dc.format.mimetype"))

        if not mimetypes:
            item_mimetype_rows.append({
                "item_id": item_id,
                "mimetype": "missing_mimetype"
            })
        else:
            for mimetype in mimetypes:
                item_mimetype_rows.append({
                    "item_id": item_id,
                    "mimetype": mimetype
                })

df_item_mimetypes_from_oai = pd.DataFrame(item_mimetype_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item = pd.DataFrame(downloads_by_item_rows)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> MIME TYPE MAP
# =========================================================

if not df_downloads_by_item.empty and not df_item_mimetypes_from_oai.empty:
    df_downloads_by_item_mimetype = df_downloads_by_item.merge(
        df_item_mimetypes_from_oai,
        on="item_id",
        how="left"
    )

    df_downloads_by_item_mimetype["mimetype"] = (
        df_downloads_by_item_mimetype["mimetype"]
        .fillna("missing_mimetype")
    )

    df_downloads_by_mimetype = (
        df_downloads_by_item_mimetype
        .groupby("mimetype", dropna=False)
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=(
                "item_id",
                lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
            ),
            bitstream_ids=("bitstream_ids", join_unique_values)
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_mimetype = pd.DataFrame(
        columns=[
            "mimetype",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_mimetype["core_name"] = statistics_core
df_downloads_by_mimetype["statistics_core"] = statistics_core
df_downloads_by_mimetype["oai_core"] = oai_core
df_downloads_by_mimetype["object_type"] = "0"
df_downloads_by_mimetype["object_type_label"] = "bitstream"
df_downloads_by_mimetype["statistics_type"] = "view"
df_downloads_by_mimetype["bundle_name"] = "ORIGINAL"
df_downloads_by_mimetype["bot_filter"] = "excluded"
df_downloads_by_mimetype["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    "AND -isBot:true FACET owningItem; OAI core used for item.id -> metadata.dc.format.mimetype mapping"
)

df_downloads_by_mimetype["reference_month"] = REPORT_MONTH
df_downloads_by_mimetype["environment"] = ENVIRONMENT
df_downloads_by_mimetype["source"] = "solr"
df_downloads_by_mimetype["metric_definition"] = (
    "bitstream downloads grouped by MIME type. Downloads are extracted from the Solr "
    "statistics core and aggregated by owningItem; MIME type information is obtained "
    "from the Solr OAI core using item.id and metadata.dc.format.mimetype. Bot events are excluded."
)

output_downloads_by_mimetype = (
    export_path("solr_downloads_by_mimetype.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_mimetype.to_csv(
    output_downloads_by_mimetype,
    index=False
)

print(f"MIME type groups with downloads: {len(df_downloads_by_mimetype)}")
print(
    "Total downloads by MIME type excluding bots: "
    f"{df_downloads_by_mimetype['downloads_count'].sum() if not df_downloads_by_mimetype.empty else 0}"
)

display(df_downloads_by_mimetype.head())

print(f"File saved to: {output_downloads_by_mimetype.resolve()}")

### Downloads by MIME type in the reference month

In [ ]:
# =========================================================
# DOWNLOADS BY MIME TYPE IN THE REFERENCE MONTH
# =========================================================

# - mimetype
# - downloads_count
# - downloaded_items_count
# - downloaded_item_ids
# - bitstream_ids
# - core_name
# - statistics_core
# - oai_core
# - object_type
# - object_type_label
# - statistics_type
# - bundle_name
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")
oai_core = SOLR_CORES.get("oai", "oai")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"


def normalize_solr_list(value):
    """
    Normalizes Solr scalar/list values into a Python list.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(v)
            for v in value
            if v is not None and str(v).strip() != ""
        ]

    if str(value).strip() == "":
        return []

    return [str(value)]


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: BUILD ITEM -> MIME TYPE MAP FROM OAI CORE
# =========================================================

response_total_oai_records_count = solr_select(
    oai_core,
    {
        "q": "*:*",
        "rows": 0,
        "wt": "json"
    }
)

total_oai_records = (
    response_total_oai_records_count
    .get("response", {})
    .get("numFound", 0)
)

rows_per_page = 1000
item_mimetype_rows = []

for start in range(0, total_oai_records, rows_per_page):
    response_oai_records = solr_select(
        oai_core,
        {
            "q": "*:*",
            "rows": rows_per_page,
            "start": start,
            "fl": "*",
            "wt": "json"
        }
    )

    docs = (
        response_oai_records
        .get("response", {})
        .get("docs", [])
    )

    for doc in docs:
        item_id = doc.get("item.id")
        mimetypes = normalize_solr_list(doc.get("metadata.dc.format.mimetype"))

        if not mimetypes:
            item_mimetype_rows.append({
                "item_id": item_id,
                "mimetype": "missing_mimetype"
            })
        else:
            for mimetype in mimetypes:
                item_mimetype_rows.append({
                    "item_id": item_id,
                    "mimetype": mimetype
                })

df_item_mimetypes_from_oai = pd.DataFrame(item_mimetype_rows)

# =========================================================
# STEP 2: GET DOWNLOADS BY ITEM FROM STATISTICS CORE
# =========================================================

response_downloads_by_item_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values = (
    response_downloads_by_item_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_reference_month_rows = []

for i in range(0, len(facet_values), 2):
    item_id = facet_values[i]
    downloads_count = facet_values[i + 1]

    response_item_bitstreams_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                time_filter,
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams_reference_month
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_reference_month_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item_reference_month = pd.DataFrame(
    downloads_by_item_reference_month_rows
)

# =========================================================
# STEP 3: JOIN DOWNLOADS WITH ITEM -> MIME TYPE MAP
# =========================================================

if (
    not df_downloads_by_item_reference_month.empty
    and not df_item_mimetypes_from_oai.empty
):
    df_downloads_by_item_mimetype_reference_month = (
        df_downloads_by_item_reference_month
        .merge(
            df_item_mimetypes_from_oai,
            on="item_id",
            how="left"
        )
    )

    df_downloads_by_item_mimetype_reference_month["mimetype"] = (
        df_downloads_by_item_mimetype_reference_month["mimetype"]
        .fillna("missing_mimetype")
    )

    df_downloads_by_mimetype_reference_month = (
        df_downloads_by_item_mimetype_reference_month
        .groupby("mimetype", dropna=False)
        .agg(
            downloads_count=("downloads_count", "sum"),
            downloaded_items_count=("item_id", "nunique"),
            downloaded_item_ids=(
                "item_id",
                lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
            ),
            bitstream_ids=("bitstream_ids", join_unique_values)
        )
        .reset_index()
        .sort_values("downloads_count", ascending=False)
    )
else:
    df_downloads_by_mimetype_reference_month = pd.DataFrame(
        columns=[
            "mimetype",
            "downloads_count",
            "downloaded_items_count",
            "downloaded_item_ids",
            "bitstream_ids"
        ]
    )

df_downloads_by_mimetype_reference_month["core_name"] = statistics_core
df_downloads_by_mimetype_reference_month["statistics_core"] = statistics_core
df_downloads_by_mimetype_reference_month["oai_core"] = oai_core
df_downloads_by_mimetype_reference_month["object_type"] = "0"
df_downloads_by_mimetype_reference_month["object_type_label"] = "bitstream"
df_downloads_by_mimetype_reference_month["statistics_type"] = "view"
df_downloads_by_mimetype_reference_month["bundle_name"] = "ORIGINAL"
df_downloads_by_mimetype_reference_month["bot_filter"] = "excluded"
df_downloads_by_mimetype_reference_month["period_start"] = (
    reference_month_start.strftime("%Y-%m-%d")
)
df_downloads_by_mimetype_reference_month["period_end"] = (
    reference_month_end.strftime("%Y-%m-%d")
)
df_downloads_by_mimetype_reference_month["query"] = (
    "statistics core: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    f"AND -isBot:true AND {time_filter} FACET owningItem; "
    "OAI core used for item.id -> metadata.dc.format.mimetype mapping"
)

df_downloads_by_mimetype_reference_month["reference_month"] = REPORT_MONTH
df_downloads_by_mimetype_reference_month["environment"] = ENVIRONMENT
df_downloads_by_mimetype_reference_month["source"] = "solr"
df_downloads_by_mimetype_reference_month["metric_definition"] = (
    "bitstream downloads during the reference month grouped by MIME type. Downloads "
    "are extracted from the Solr statistics core and aggregated by owningItem; MIME type "
    "information is obtained from the Solr OAI core using item.id and metadata.dc.format.mimetype. "
    "Bot events are excluded."
)

output_downloads_by_mimetype_reference_month = (
    export_path("solr_downloads_by_mimetype_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_downloads_by_mimetype_reference_month.to_csv(
    output_downloads_by_mimetype_reference_month,
    index=False
)

print(
    "MIME type groups with downloads in the reference month: "
    f"{len(df_downloads_by_mimetype_reference_month)}"
)

print(
    "Total downloads by MIME type in the reference month excluding bots: "
    f"{df_downloads_by_mimetype_reference_month['downloads_count'].sum() if not df_downloads_by_mimetype_reference_month.empty else 0}"
)

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_downloads_by_mimetype_reference_month.head())

print(f"File saved to: {output_downloads_by_mimetype_reference_month.resolve()}")

### Views / Downloads ratio by item

In [ ]:
# =========================================================
# VIEWS / DOWNLOADS RATIO BY ITEM
# =========================================================

# - item_id
# - views_count
# - downloads_count
# - views_downloads_ratio
# - downloads_views_ratio
# - bitstream_ids
# - core_name
# - statistics_core
# - object_type_views
# - object_type_downloads
# - statistics_type
# - bundle_name
# - bot_filter
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")


def join_unique_values(values):
    """
    Joins unique non-empty values from a pandas Series.
    Values may already contain '; '-separated strings.
    """

    unique_values = set()

    for value in values.dropna().astype(str):
        for part in value.split("; "):
            if part.strip() != "":
                unique_values.add(part.strip())

    return "; ".join(sorted(unique_values))


# =========================================================
# STEP 1: ITEM VIEWS
# =========================================================

response_views_by_item = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:2",
            "statistics_type:view",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values_views = (
    response_views_by_item
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

views_by_item_rows = []

for i in range(0, len(facet_values_views), 2):
    views_by_item_rows.append({
        "item_id": facet_values_views[i],
        "views_count": facet_values_views[i + 1]
    })

df_views_by_item = pd.DataFrame(
    views_by_item_rows,
    columns=["item_id", "views_count"]
)


# =========================================================
# STEP 2: BITSTREAM DOWNLOADS AGGREGATED BY ITEM
# =========================================================

response_downloads_by_item = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true"
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values_downloads = (
    response_downloads_by_item
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_rows = []

for i in range(0, len(facet_values_downloads), 2):
    item_id = facet_values_downloads[i]
    downloads_count = facet_values_downloads[i + 1]

    response_item_bitstreams = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item = pd.DataFrame(
    downloads_by_item_rows,
    columns=["item_id", "downloads_count", "bitstream_ids"]
)


# =========================================================
# STEP 3: JOIN VIEWS AND DOWNLOADS
# =========================================================

df_views_downloads_ratio_by_item = pd.merge(
    df_views_by_item,
    df_downloads_by_item,
    on="item_id",
    how="outer"
)

df_views_downloads_ratio_by_item["views_count"] = (
    df_views_downloads_ratio_by_item["views_count"]
    .fillna(0)
    .astype(int)
)

df_views_downloads_ratio_by_item["downloads_count"] = (
    df_views_downloads_ratio_by_item["downloads_count"]
    .fillna(0)
    .astype(int)
)

df_views_downloads_ratio_by_item["bitstream_ids"] = (
    df_views_downloads_ratio_by_item["bitstream_ids"]
    .fillna("")
)

df_views_downloads_ratio_by_item["views_downloads_ratio"] = (
    df_views_downloads_ratio_by_item.apply(
        lambda row: (
            round(row["views_count"] / row["downloads_count"], 4)
            if row["downloads_count"] > 0
            else None
        ),
        axis=1
    )
)

df_views_downloads_ratio_by_item["downloads_views_ratio"] = (
    df_views_downloads_ratio_by_item.apply(
        lambda row: (
            round(row["downloads_count"] / row["views_count"], 4)
            if row["views_count"] > 0
            else None
        ),
        axis=1
    )
)

df_views_downloads_ratio_by_item["core_name"] = statistics_core
df_views_downloads_ratio_by_item["statistics_core"] = statistics_core
df_views_downloads_ratio_by_item["object_type_views"] = "2"
df_views_downloads_ratio_by_item["object_type_downloads"] = "0"
df_views_downloads_ratio_by_item["statistics_type"] = "view"
df_views_downloads_ratio_by_item["bundle_name"] = "ORIGINAL"
df_views_downloads_ratio_by_item["bot_filter"] = "excluded"
df_views_downloads_ratio_by_item["query"] = (
    "views: type:2 AND statistics_type:view AND -isBot:true FACET id; "
    "downloads: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    "AND -isBot:true FACET owningItem"
)

df_views_downloads_ratio_by_item["reference_month"] = REPORT_MONTH
df_views_downloads_ratio_by_item["environment"] = ENVIRONMENT
df_views_downloads_ratio_by_item["source"] = "solr"
df_views_downloads_ratio_by_item["metric_definition"] = (
    "views/downloads ratio by item. Item views are based on type:2 view events; "
    "downloads are based on bitstream type:0 view/download events aggregated by owningItem, "
    "restricted to the ORIGINAL bundle and excluding bot events"
)

df_views_downloads_ratio_by_item = (
    df_views_downloads_ratio_by_item
    .sort_values(
        ["downloads_count", "views_count"],
        ascending=[False, False]
    )
)

output_views_downloads_ratio_by_item = (
    export_path("solr_views_downloads_ratio_by_item.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_views_downloads_ratio_by_item.to_csv(
    output_views_downloads_ratio_by_item,
    index=False
)

print(
    "Items with views and/or downloads: "
    f"{len(df_views_downloads_ratio_by_item)}"
)

print(
    "Total item views excluding bots: "
    f"{df_views_downloads_ratio_by_item['views_count'].sum()}"
)

print(
    "Total bitstream downloads excluding bots: "
    f"{df_views_downloads_ratio_by_item['downloads_count'].sum()}"
)

display(df_views_downloads_ratio_by_item.head())

print(f"File saved to: {output_views_downloads_ratio_by_item.resolve()}")

### Views / Downloads ratio by item in the reference month

In [ ]:
# =========================================================
# VIEWS / DOWNLOADS RATIO BY ITEM IN THE REFERENCE MONTH
# =========================================================

# - item_id
# - views_count
# - downloads_count
# - views_downloads_ratio
# - downloads_views_ratio
# - bitstream_ids
# - core_name
# - statistics_core
# - object_type_views
# - object_type_downloads
# - statistics_type
# - bundle_name
# - bot_filter
# - period_start
# - period_end
# - query
# - reference_month
# - environment
# - source
# - metric_definition

statistics_core = SOLR_CORES.get("statistics", "statistics")

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start_solr = reference_month_start.strftime("%Y-%m-%dT00:00:00Z")
period_end_solr = reference_month_end.strftime("%Y-%m-%dT00:00:00Z")

time_filter = f"time:[{period_start_solr} TO {period_end_solr}}}"


# =========================================================
# STEP 1: ITEM VIEWS IN THE REFERENCE MONTH
# =========================================================

response_views_by_item_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:2",
            "statistics_type:view",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "id",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values_views = (
    response_views_by_item_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("id", [])
)

views_by_item_reference_month_rows = []

for i in range(0, len(facet_values_views), 2):
    views_by_item_reference_month_rows.append({
        "item_id": facet_values_views[i],
        "views_count": facet_values_views[i + 1]
    })

df_views_by_item_reference_month = pd.DataFrame(
    views_by_item_reference_month_rows,
    columns=["item_id", "views_count"]
)


# =========================================================
# STEP 2: BITSTREAM DOWNLOADS AGGREGATED BY ITEM IN THE REFERENCE MONTH
# =========================================================

response_downloads_by_item_reference_month = solr_select(
    statistics_core,
    {
        "q": "*:*",
        "fq": [
            "type:0",
            "statistics_type:view",
            "bundleName:ORIGINAL",
            "-isBot:true",
            time_filter
        ],
        "rows": 0,
        "facet": "true",
        "facet.field": "owningItem",
        "facet.mincount": 1,
        "facet.sort": "count",
        "facet.limit": -1,
        "wt": "json"
    }
)

facet_values_downloads = (
    response_downloads_by_item_reference_month
    .get("facet_counts", {})
    .get("facet_fields", {})
    .get("owningItem", [])
)

downloads_by_item_reference_month_rows = []

for i in range(0, len(facet_values_downloads), 2):
    item_id = facet_values_downloads[i]
    downloads_count = facet_values_downloads[i + 1]

    response_item_bitstreams_reference_month = solr_select(
        statistics_core,
        {
            "q": "*:*",
            "fq": [
                "type:0",
                "statistics_type:view",
                "bundleName:ORIGINAL",
                "-isBot:true",
                time_filter,
                f'owningItem:"{item_id}"'
            ],
            "rows": 0,
            "facet": "true",
            "facet.field": "id",
            "facet.mincount": 1,
            "facet.sort": "count",
            "facet.limit": -1,
            "wt": "json"
        }
    )

    bitstream_facet_values = (
        response_item_bitstreams_reference_month
        .get("facet_counts", {})
        .get("facet_fields", {})
        .get("id", [])
    )

    bitstream_ids = [
        str(bitstream_facet_values[j])
        for j in range(0, len(bitstream_facet_values), 2)
    ]

    downloads_by_item_reference_month_rows.append({
        "item_id": item_id,
        "downloads_count": downloads_count,
        "bitstream_ids": "; ".join(bitstream_ids)
    })

df_downloads_by_item_reference_month = pd.DataFrame(
    downloads_by_item_reference_month_rows,
    columns=["item_id", "downloads_count", "bitstream_ids"]
)


# =========================================================
# STEP 3: JOIN VIEWS AND DOWNLOADS
# =========================================================

df_views_downloads_ratio_by_item_reference_month = pd.merge(
    df_views_by_item_reference_month,
    df_downloads_by_item_reference_month,
    on="item_id",
    how="outer"
)

df_views_downloads_ratio_by_item_reference_month["views_count"] = (
    df_views_downloads_ratio_by_item_reference_month["views_count"]
    .fillna(0)
    .astype(int)
)

df_views_downloads_ratio_by_item_reference_month["downloads_count"] = (
    df_views_downloads_ratio_by_item_reference_month["downloads_count"]
    .fillna(0)
    .astype(int)
)

df_views_downloads_ratio_by_item_reference_month["bitstream_ids"] = (
    df_views_downloads_ratio_by_item_reference_month["bitstream_ids"]
    .fillna("")
)

df_views_downloads_ratio_by_item_reference_month["views_downloads_ratio"] = (
    df_views_downloads_ratio_by_item_reference_month.apply(
        lambda row: (
            round(row["views_count"] / row["downloads_count"], 4)
            if row["downloads_count"] > 0
            else None
        ),
        axis=1
    )
)

df_views_downloads_ratio_by_item_reference_month["downloads_views_ratio"] = (
    df_views_downloads_ratio_by_item_reference_month.apply(
        lambda row: (
            round(row["downloads_count"] / row["views_count"], 4)
            if row["views_count"] > 0
            else None
        ),
        axis=1
    )
)

df_views_downloads_ratio_by_item_reference_month["core_name"] = statistics_core
df_views_downloads_ratio_by_item_reference_month["statistics_core"] = statistics_core
df_views_downloads_ratio_by_item_reference_month["object_type_views"] = "2"
df_views_downloads_ratio_by_item_reference_month["object_type_downloads"] = "0"
df_views_downloads_ratio_by_item_reference_month["statistics_type"] = "view"
df_views_downloads_ratio_by_item_reference_month["bundle_name"] = "ORIGINAL"
df_views_downloads_ratio_by_item_reference_month["bot_filter"] = "excluded"
df_views_downloads_ratio_by_item_reference_month["period_start"] = (
    reference_month_start.strftime("%Y-%m-%d")
)
df_views_downloads_ratio_by_item_reference_month["period_end"] = (
    reference_month_end.strftime("%Y-%m-%d")
)
df_views_downloads_ratio_by_item_reference_month["query"] = (
    "views: type:2 AND statistics_type:view AND -isBot:true "
    f"AND {time_filter} FACET id; "
    "downloads: type:0 AND statistics_type:view AND bundleName:ORIGINAL "
    f"AND -isBot:true AND {time_filter} FACET owningItem"
)

df_views_downloads_ratio_by_item_reference_month["reference_month"] = REPORT_MONTH
df_views_downloads_ratio_by_item_reference_month["environment"] = ENVIRONMENT
df_views_downloads_ratio_by_item_reference_month["source"] = "solr"
df_views_downloads_ratio_by_item_reference_month["metric_definition"] = (
    "views/downloads ratio by item during the reference month. Item views are based "
    "on type:2 view events; downloads are based on bitstream type:0 view/download "
    "events aggregated by owningItem, restricted to the ORIGINAL bundle and excluding bot events"
)

df_views_downloads_ratio_by_item_reference_month = (
    df_views_downloads_ratio_by_item_reference_month
    .sort_values(
        ["downloads_count", "views_count"],
        ascending=[False, False]
    )
)

output_views_downloads_ratio_by_item_reference_month = (
    export_path("solr_views_downloads_ratio_by_item_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
df_views_downloads_ratio_by_item_reference_month.to_csv(
    output_views_downloads_ratio_by_item_reference_month,
    index=False
)

print(
    "Items with views and/or downloads in the reference month: "
    f"{len(df_views_downloads_ratio_by_item_reference_month)}"
)

print(
    "Total item views in the reference month excluding bots: "
    f"{df_views_downloads_ratio_by_item_reference_month['views_count'].sum()}"
)

print(
    "Total bitstream downloads in the reference month excluding bots: "
    f"{df_views_downloads_ratio_by_item_reference_month['downloads_count'].sum()}"
)

print(
    "Reference period: "
    f"{reference_month_start.strftime('%Y-%m-%d')} "
    f"to {reference_month_end.strftime('%Y-%m-%d')}"
)

display(df_views_downloads_ratio_by_item_reference_month.head())

print(f"File saved to: {output_views_downloads_ratio_by_item_reference_month.resolve()}")